<a href="https://colab.research.google.com/github/Rollet89/Rollet89/blob/main/Mapungubwe%20MISTRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =============================================================================
# MISTRA 2026 PGM-POWERED HYDROGEN ECONOMY — EXECUTION DASHBOARD
# Single-cell Python project: synthetic data → PyTorch → Plotly → MCP/LLM
# =============================================================================
# pip install numpy pandas plotly torch scikit-learn networkx kaleido jupyter
# (Figma MCP / Kubeflow / SCADA / HMI are simulated via stubs for demonstration)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import networkx as nx
import json, random, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# =============================================================================
# 1. SYNTHETIC DATA GENERATION — PGM-HYDROGEN VALUE CHAIN
# =============================================================================

# ---- 1a. PGM Production Time Series (quarterly, 6E PGM oz) ----
quarters = pd.date_range(start='2023-01-01', periods=16, freq='Q')
pgm_production = {
    'Quarter': quarters,
    '6E_PGM_oz': [34500, 36200, 37100, 35800, 37400, 38100, 39000, 38579,
                  39200, 40100, 41000, 40500, 41500, 42200, 43000, 43800],
    'Pt_share_pct': [59.0, 59.2, 58.8, 59.1, 59.3, 59.0, 58.7, 59.0,
                     59.2, 59.5, 59.3, 59.1, 59.4, 59.6, 59.5, 59.7],
    'Pd_share_pct': [30.0, 29.8, 30.1, 29.9, 29.7, 30.0, 30.2, 30.0,
                     29.8, 29.6, 29.8, 30.0, 29.7, 29.5, 29.6, 29.4],
    'Rh_share_pct': [9.0, 9.0, 9.1, 9.0, 9.0, 9.0, 9.1, 9.0,
                     9.0, 8.9, 8.9, 8.9, 8.9, 8.9, 8.9, 8.9]
}
df_pgm = pd.DataFrame(pgm_production)
df_pgm['Pt_oz'] = df_pgm['6E_PGM_oz'] * df_pgm['Pt_share_pct'] / 100
df_pgm['Revenue_USDm'] = df_pgm['6E_PGM_oz'] * 1650 / 1e6  # ~Pt price $1,650/oz

# ---- 1b. First Wave Green Hydrogen Projects (6 projects) ----
projects = pd.DataFrame({
    'Project': ['Coega Green Ammonia', 'Saldanha H2-DRI', "Phelan Green e-SAF",
                'Prieska Power Reserve', 'Green e-Fuels Methanol', 'Green H2 Solutions'],
    'Province': ['Eastern Cape', 'Western Cape', 'Western Cape',
                 'Northern Cape', 'Gauteng', 'Eastern Cape'],
    'Stage': ['Pre-FS', 'FS', 'FID', 'Development', 'PFS', 'Construction'],
    'Capacity_MW': [500, 800, 300, 200, 150, 50],
    'Capex_USDm': [1200, 2100, 750, 450, 380, 120],
    'Jobs_Potential': [1800, 3200, 950, 600, 480, 180],
    'PGM_Intensity_kg_per_MW': [0.18, 0.22, 0.15, 0.12, 0.14, 0.10]
})

# ---- 1c. Investment Pipeline & ROI Scenarios ----
years = list(range(2025, 2036))
investment_scenarios = pd.DataFrame({
    'Year': years,
    'Conservative_USDm': np.cumsum(np.random.normal(180, 40, len(years))).clip(0).round(0),
    'Base_USDm': np.cumsum(np.random.normal(350, 60, len(years))).clip(0).round(0),
    'Accelerated_USDm': np.cumsum(np.random.normal(520, 80, len(years))).clip(0).round(0)
})
investment_scenarios['Base_ROI_pct'] = np.random.normal(14, 3, len(years)).round(1)
investment_scenarios['Accelerated_ROI_pct'] = np.random.normal(18, 4, len(years)).round(1)

# ---- 1d. Skills Demand Matrix (by occupation group) ----
occupations = ['Fuel Cell Engineers', 'Electrolyser Technicians', 'PGM Catalysis Scientists',
               'Hydrogen Safety Specialists', 'Project Finance Analysts', 'Plant Operators']
skills_matrix = pd.DataFrame({
    'Occupation': occupations,
    'Current_Workforce': np.random.randint(150, 800, len(occupations)),
    'Demand_2030': np.random.randint(500, 3000, len(occupations)),
    'Training_Institutions': np.random.randint(2, 12, len(occupations)),
    'PGM_Linkage_Score': np.random.uniform(0.4, 0.95, len(occupations)).round(2)
})

# ---- 1e. Collaboration Network (government, industry, academia) ----
collab_edges = [
    ('MISTRA', 'DTIC'), ('MISTRA', 'DEE'), ('MISTRA', 'Isondo Precious Metals'),
    ('DTIC', 'IDC'), ('DTIC', 'HySA'), ('DEE', 'Phelan Green'),
    ('IDC', 'Phelan Green'), ('HySA', 'North-West University'),
    ('HySA', 'University of Cape Town'), ('Isondo Precious Metals', 'Coega Green Ammonia'),
    ('Phelan Green', 'Saldanha H2-DRI'), ('Coega Green Ammonia', 'Green H2 Solutions'),
    ('Prieska Power Reserve', 'Green e-Fuels Methanol'), ('MISTRA', 'Creamer Media')
]
G = nx.Graph()
G.add_edges_from(collab_edges)

# ---- 1f. Wafer-thickness / membrane-electrode-assembly (MEA) contour ----
# Simulates Pt catalyst layer thickness uniformity across a 200 mm wafer
x_wafer = np.linspace(-1, 1, 60)
y_wafer = np.linspace(-1, 1, 60)
X, Y = np.meshgrid(x_wafer, y_wafer)
Z_wafer = (0.08 * np.exp(-(X**2 + Y**2) / 0.3)
           + 0.02 * np.sin(8 * X) * np.cos(6 * Y)
           + 0.01 * np.random.randn(*X.shape)).clip(0.0, 0.15)

# ---- 1g. Financial / ROI table ----
roi_table = pd.DataFrame({
    'Project': projects['Project'],
    'IRR_pct': np.random.normal(15, 4, len(projects)).round(1),
    'NPV_USDm': (projects['Capex_USDm'] * np.random.uniform(0.1, 0.4, len(projects))).round(0),
    'Payback_Years': np.random.randint(6, 12, len(projects)),
    'Risk_Score': np.random.uniform(0.2, 0.8, len(projects)).round(2)
})

# =============================================================================
# 2. PYTORCH MODEL — HYDROGEN READINESS INDEX (HRI)
# =============================================================================
# Predict project "readiness" from synthetic features.
X_features = projects[['Capacity_MW', 'Capex_USDm', 'Jobs_Potential',
                       'PGM_Intensity_kg_per_MW']].values
# Synthesise a latent readiness score (for training demonstration only)
y_target = (0.4 * X_features[:, 0] / 1000
            + 0.3 * X_features[:, 1] / 2000
            + 0.2 * X_features[:, 2] / 3000
            + 0.1 * X_features[:, 3] / 0.2
            + np.random.normal(0, 0.05, len(X_features)))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_target, test_size=0.33, random_state=42
)

class ReadinessNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

model = ReadinessNet(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    pred = model(torch.FloatTensor(X_train)).squeeze()
    loss = criterion(pred, torch.FloatTensor(y_train))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_pred = model(torch.FloatTensor(X_test)).squeeze().numpy()

# =============================================================================
# 3. LLM + MCP SERVER SIMULATION (Decision-Support Node)
# =============================================================================
class MCPClientSimulator:
    """Simulates an MCP client that queries a reasoning LLM for
    PGM-hydrogen policy insights."""
    def __init__(self):
        self.knowledge = {
            'bottleneck': 'Skills gap in electrolyser and fuel-cell manufacturing',
            'priority': 'Unlock FID-stage projects and localise MEA production',
            'policy': 'Blended finance + PGM beneficiation tax incentive'
        }
    def query(self, prompt: str) -> str:
        # Placeholder for real LLM API (OpenAI / Anthropic / local Llama)
        return (f"[LLM-MCP] Strategic response to '{prompt}': "
                f"Focus on {self.knowledge['bottleneck']}. "
                f"Recommended action: {self.knowledge['priority']}.")

mcp = MCPClientSimulator()
llm_output = mcp.query("How do we move from strategy to execution?")

# =============================================================================
# 4. FIGMA MCP SERVER STUB (design-token hand-off)
# =============================================================================
def figma_mcp_export(chart_config: dict) -> str:
    """Serialises Plotly chart layout into a Figma-compatible token JSON."""
    return json.dumps({"figma_mcp": "ready", "tokens": chart_config}, indent=2)

figma_tokens = figma_mcp_export({"primary": "#00A3E0", "font": "Inter",
                                 "chart_bg": "#0F172A"})

# =============================================================================
# 5. VISUALISATION SUITE
# =============================================================================

# ---- 5a. Basic + Statistical: Time series subplot (PGM production & revenue) ----
fig_ts = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("6E PGM Production (oz)", "Implied Revenue (USD m)"))
fig_ts.add_trace(go.Scatter(x=df_pgm['Quarter'], y=df_pgm['6E_PGM_oz'],
                            mode='lines+markers', name='6E PGM oz',
                            line=dict(color='#00A3E0', width=2)), row=1, col=1)
fig_ts.add_trace(go.Bar(x=df_pgm['Quarter'], y=df_pgm['Revenue_USDm'],
                         name='Revenue (USD m)', marker_color='#F4A261'), row=2, col=1)
fig_ts.update_layout(title="PGM Production & Revenue Trend (Synthetic, 2023–2026)",
                     template='plotly_dark', height=500, showlegend=True)

# ---- 5b. Scatter plot matrix (project features) ----
fig_spm = px.scatter_matrix(
    projects, dimensions=['Capacity_MW', 'Capex_USDm', 'Jobs_Potential'],
    color='Stage', title="Project Scatter Matrix — Capacity, Capex, Jobs",
    template='plotly_dark', height=500
)

# ---- 5c. Scatter map (projects geographically) ----
project_coords = pd.DataFrame({
    'Project': projects['Project'],
    'lat': [-33.0, -33.0, -33.0, -29.7, -26.2, -33.0],
    'lon': [25.6, 18.0, 18.0, 22.7, 28.0, 25.6],
    'Capacity_MW': projects['Capacity_MW']
})
fig_map = px.scatter_mapbox(
    project_coords, lat='lat', lon='lon', size='Capacity_MW',
    hover_name='Project', color='Capacity_MW', zoom=4,
    mapbox_style='carto-darkmatter',
    title="First Wave Green Hydrogen Projects — Geographic Distribution",
    template='plotly_dark', height=500
)

# ---- 5d. Tree map (project capex hierarchy) ----
fig_treemap = px.treemap(
    projects, path=['Province', 'Project'], values='Capex_USDm',
    color='Jobs_Potential', color_continuous_scale='Viridis',
    title="Capex Allocation by Province & Project (Tree Map)",
    template='plotly_dark', height=500
)

# ---- 5e. Sunburst (skills demand hierarchy) ----
skills_hierarchy = pd.DataFrame({
    'Sector': ['Hydrogen Economy'] * len(occupations),
    'Occupation': occupations,
    'Demand': skills_matrix['Demand_2030']
})
fig_sunburst = px.sunburst(
    skills_hierarchy, path=['Sector', 'Occupation'], values='Demand',
    color='Demand', color_continuous_scale='Plasma',
    title="Skills Demand Hierarchy (Sunburst)",
    template='plotly_dark', height=500
)

# ---- 5f. Sankey diagram (investment flow: sources → projects → outcomes) ----
fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(label=["Government", "Private Equity", "DFIs",
                      "Coega NH3", "Saldanha DRI", "Phelan e-SAF",
                      "Jobs", "Industrialisation", "Exports"],
              color=["#00A3E0", "#F4A261", "#2A9D8F",
                     "#E76F51", "#E9C46A", "#264653",
                     "#A8DADC", "#457B9D", "#1D3557"]),
    link=dict(source=[0, 0, 1, 1, 2, 2, 3, 4, 5],
              target=[3, 4, 4, 5, 3, 5, 6, 7, 8],
              value=[300, 200, 400, 250, 150, 180, 1200, 900, 700])
)])
fig_sankey.update_layout(title="Investment Flow — Sankey (Synthetic USD m)",
                          template='plotly_dark', height=500)

# ---- 5g. Heat map (skills demand vs current workforce) ----
heat_data = skills_matrix.set_index('Occupation')[['Current_Workforce', 'Demand_2030']].T
fig_heat = px.imshow(
    heat_data, text_auto=True, aspect='auto',
    color_continuous_scale='YlOrRd',
    title="Skills Gap Heat Map — Current Workforce vs 2030 Demand",
    template='plotly_dark', height=400
)

# ---- 5h. 3D scatter plot (project risk-return-capacity) ----
fig_3d_scatter = px.scatter_3d(
    roi_table, x='IRR_pct', y='NPV_USDm', z='Risk_Score',
    color='Payback_Years', size='NPV_USDm', hover_name='Project',
    title="3D Project Risk–Return–Payback Space",
    template='plotly_dark', height=500
)

# ---- 5i. 3D yield curve surface (synthetic bond-yield analogue for project finance) ----
maturities = np.arange(1, 11)
spreads = np.arange(0, 5, 0.5)
M, S = np.meshgrid(maturities, spreads)
Z_yield = 8 + 0.3 * M + 0.5 * S + 0.02 * M * S + np.random.randn(*M.shape) * 0.1
fig_3d_yield = go.Figure(data=[go.Surface(z=Z_yield, x=maturities, y=spreads,
                                           colorscale='Viridis')])
fig_3d_yield.update_layout(
    title="3D Yield-Curve Surface — Project Finance Spread Scenarios",
    scene=dict(xaxis_title='Maturity (yrs)', yaxis_title='Spread (%)',
               zaxis_title='Yield (%)'), template='plotly_dark', height=500
)

# ---- 5j. Wafer-thickness contour (imshow + contour) ----
fig_wafer = go.Figure(data=go.Contour(
    z=Z_wafer, x=x_wafer, y=y_wafer, colorscale='Jet',
    contours=dict(showlabels=True), colorbar=dict(title='Thickness (mm)')
))
fig_wafer.update_layout(
    title="MEA / Pt Catalyst Layer Thickness Contour (Synthetic Wafer)",
    template='plotly_dark', height=500
)

# ---- 5k. Radar chart (collaboration strength by stakeholder) ----
stakeholders = ['Government', 'Industry', 'Academia', 'Finance', 'Communities']
radar_scores = [0.85, 0.72, 0.68, 0.60, 0.45]
fig_radar = go.Figure(data=go.Scatterpolar(
    r=radar_scores, theta=stakeholders, fill='toself',
    name='Collaboration Index', line_color='#00A3E0'
))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Stakeholder Collaboration Radar (Synthetic Index)",
    template='plotly_dark', height=400
)

# ---- 5l. Network graph (collaboration ecosystem) ----
pos = nx.spring_layout(G, seed=42)
edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]; x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]

fig_network = go.Figure()
fig_network.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                                  line=dict(width=0.8, color='#888')))
fig_network.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                                  text=list(G.nodes()), textposition='top center',
                                  marker=dict(size=14, color='#00A3E0')))
fig_network.update_layout(
    title="PGM-Hydrogen Collaboration Network (Synthetic Edges)",
    template='plotly_dark', height=500, showlegend=False
)

# ---- 5m. Financial / ROI bar chart ----
fig_roi = px.bar(
    roi_table.melt(id_vars='Project', value_vars=['IRR_pct', 'NPV_USDm']),
    x='Project', y='value', color='variable', barmode='group',
    title="Project ROI & NPV Comparison", template='plotly_dark', height=450
)

# ---- 5n. Investment scenario line chart ----
fig_invest = go.Figure()
for col, color in [('Conservative_USDm', '#E9C46A'),
                    ('Base_USDm', '#F4A261'),
                    ('Accelerated_USDm', '#E76F51')]:
    fig_invest.add_trace(go.Scatter(x=investment_scenarios['Year'],
                                     y=investment_scenarios[col],
                                     mode='lines+markers', name=col,
                                     line=dict(width=2.5, color=color)))
fig_invest.update_layout(
    title="Hydrogen Investment Scenarios (Synthetic, 2025–2035)",
    template='plotly_dark', height=450
)

# =============================================================================
# 6. JUPYTER WIDGETS / CUSTOM CONTROL (simulated inline)
# =============================================================================
# In a live notebook you would use ipywidgets.interact; here we simulate
# the control logic so the cell remains self-contained.
def simulate_widget_response(scenario='Base'):
    return f"Widget control: scenario = {scenario} → chart refresh queued."

widget_response = simulate_widget_response('Accelerated')

# =============================================================================
# 7. DISPLAY / EXPORT SUMMARY
# =============================================================================
print("=" * 70)
print("MISTRA 2026 PGM-HYDROGEN EXECUTION DASHBOARD — BUILD COMPLETE")
print("=" * 70)
print(f"PGM records: {len(df_pgm)} | Projects: {len(projects)}")
print(f"PyTorch final loss: {losses[-1]:.6f}")
print(f"LLM-MCP insight: {llm_output}")
print(f"Figma tokens ready: {figma_tokens[:60]}...")
print(f"Widget response: {widget_response}")
print("\n— Rendered figures —")
for name in ['fig_ts', 'fig_spm', 'fig_map', 'fig_treemap', 'fig_sunburst',
             'fig_sankey', 'fig_heat', 'fig_3d_scatter', 'fig_3d_yield',
             'fig_wafer', 'fig_radar', 'fig_network', 'fig_roi', 'fig_invest']:
    print(f"  ✓ {name}")

# Show all figures inline (Jupyter renders these automatically)
fig_ts.show()
fig_spm.show()
fig_map.show()
fig_treemap.show()
fig_sunburst.show()
fig_sankey.show()
fig_heat.show()
fig_3d_scatter.show()
fig_3d_yield.show()
fig_wafer.show()
fig_radar.show()
fig_network.show()
fig_roi.show()
fig_invest.show()

MISTRA 2026 PGM-HYDROGEN EXECUTION DASHBOARD — BUILD COMPLETE
PGM records: 16 | Projects: 6
PyTorch final loss: 0.003406
LLM-MCP insight: [LLM-MCP] Strategic response to 'How do we move from strategy to execution?': Focus on Skills gap in electrolyser and fuel-cell manufacturing. Recommended action: Unlock FID-stage projects and localise MEA production.
Figma tokens ready: {
  "figma_mcp": "ready",
  "tokens": {
    "primary": "#00A...
Widget response: Widget control: scenario = Accelerated → chart refresh queued.

— Rendered figures —
  ✓ fig_ts
  ✓ fig_spm
  ✓ fig_map
  ✓ fig_treemap
  ✓ fig_sunburst
  ✓ fig_sankey
  ✓ fig_heat
  ✓ fig_3d_scatter
  ✓ fig_3d_yield
  ✓ fig_wafer
  ✓ fig_radar
  ✓ fig_network
  ✓ fig_roi
  ✓ fig_invest


In [2]:

# =============================================================================
#  MISTRA 2026 ANNUAL PGM ROUNDTABLE — PGM-POWERED HYDROGEN ECONOMY DASHBOARD
#  Single-cell synthetic-data project
#  Stack: NumPy · Pandas · Plotly · NetworkX · Statistical/3D/Map/Network charts
#  Output: mistra_pgm_h2_dashboard.html (open in browser)
# =============================================================================
import numpy as np, pandas as pd, networkx as nx
import plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots
from plotly.io import to_html

np.random.seed(42)
FIGS = {}
def add(name, fig): FIGS[name] = fig

# =============================================================================
# 1. SYNTHETIC DATA LAYER
# =============================================================================
provinces = ['Gauteng','Limpopo','North West','Mpumalanga','Free State',
             'KwaZulu-Natal','Eastern Cape','Western Cape','Northern Cape']
prov_coords = {'Gauteng':(-26.27,28.11),'Limpopo':(-23.90,29.45),
    'North West':(-25.85,25.64),'Mpumalanga':(-25.47,30.98),
    'Free State':(-28.45,26.52),'KwaZulu-Natal':(-28.53,30.90),
    'Eastern Cape':(-32.30,26.42),'Western Cape':(-33.92,18.42),
    'Northern Cape':(-29.05,21.85)}
pgm = np.array([120,850,920,180,45,30,20,15,40])
h2  = np.array([5,8,4,6,2,3,3,4,6])
dfp = pd.DataFrame({'Province':provinces,'PGM_Reserves_Moz':pgm,'H2_Projects':h2,
                    'lat':[prov_coords[p][0] for p in provinces],
                    'lon':[prov_coords[p][1] for p in provinces]})

years = np.arange(2018,2031)
df_ts = pd.DataFrame({'Year':years})
df_ts['Investment_Rbn']  = np.clip(0.4*np.exp(0.35*(years-2018)) + np.random.normal(0,0.2,len(years)),0,None)
df_ts['Skills_Index']    = 25 + 5.8*(years-2018) + np.random.normal(0,1.5,len(years))
df_ts['Cum_Projects']    = 1.5*np.exp(0.28*(years-2018))
df_ts['Jobs_Created']    = df_ts['Cum_Projects']*120 + np.random.normal(0,20,len(years))

alloc = pd.DataFrame({
    'Category':['Electrolyzer Mfg','Fuel Cell R&D','PGM Beneficiation',
                'Renewable Energy','Infra & Ports','Skills Development',
                'H2 Storage','Regulatory'],
    'Investment_Rbn':[18.5,12.3,15.7,22.1,8.9,4.5,7.2,3.1],
    'Sector':['Manufacturing','R&D','Beneficiation','Energy','Infra','Skills','Infra','Policy']})

# 3D project scatter
n_proj = 80
df_proj = pd.DataFrame({
    'Capacity_MW':np.random.uniform(5,500,n_proj),
    'Investment_Rbn':np.random.uniform(0.1,25,n_proj),
    'Distance_Port_km':np.random.uniform(10,900,n_proj),
    'Type':np.random.choice(['Electrolysis','Fuel Cell','Storage','PGM-Mining'],n_proj),
    'Province':np.random.choice(provinces,n_proj)})

# Radar
radar_cats = ['Policy','Finance','Skills','Infrastructure','Technology','Collaboration','Market']
current = [7,5,4,5,6,5,4]
target  = [9,9,8,8,9,9,8]

# Correlation heatmap
metrics = ['Policy','Finance','Skills','Infra','Tech','Collab','Market']
M = np.random.uniform(-0.2,1,(7,7)); C = (M+M.T)/2; np.fill_diagonal(C,1)

# Wafer-thickness-style contour (skills density)
xg,yg = np.meshgrid(np.linspace(0,10,50), np.linspace(0,10,50))
Zc = np.sin(xg/2)*np.cos(yg/2)*10 + 15 + np.random.normal(0,0.3,xg.shape)

# 3D yield surface
Xs,Ys = np.meshgrid(np.linspace(0,10,40), np.linspace(0,10,40))
Zs = (1-np.exp(-((Xs-5)**2+(Ys-5)**2)/20))*8 + 2

# Financial time series
dates = pd.date_range('2024-01-31', periods=36, freq='ME')
price = 100*np.cumprod(1+np.random.normal(0.01,0.05,36))
rev   = 50 + np.arange(36)*3 + np.random.normal(0,2,36)
df_fin = pd.DataFrame({'Date':dates,'Price_Index':price,'Revenue_Rbn':rev})

# Collar plan (drill collars over an area)
df_collar = pd.DataFrame({'Easting':np.random.uniform(0,1000,60),
                          'Northing':np.random.uniform(0,1000,60),
                          'PGM_ppm':np.random.uniform(1,9,60),
                          'Depth_m':np.random.uniform(20,180,60)})

# Network (stakeholders)
G = nx.Graph()
G.add_edges_from([('Government','MISTRA'),('MISTRA','Creamer Media'),
    ('Government','Mining Cos'),('Mining Cos','Fuel Cell OEMs'),
    ('Fuel Cell OEMs','Universities'),('Banks','Government'),
    ('Banks','Mining Cos'),('Communities','Government'),
    ('SADC Partners','Government'),('Sovereign Funds','Banks'),
    ('Universities','MISTRA'),('Creamer Media','Fuel Cell OEMs'),
    ('Universities','SADC Partners'),('Communities','Mining Cos')])
pos = nx.spring_layout(G, seed=7, k=0.9)

# =============================================================================
# 2. FIGURES
# =============================================================================

# --- 2.1 MAP: PGM reserves + H2 project density -----------------------------
fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=dfp.lon, lat=dfp.lat,
    text=dfp.Province, mode='markers+text', textposition='top center',
    marker=dict(size=np.sqrt(dfp.PGM_Reserves_Moz)*3.2,
                color=dfp.H2_Projects, colorscale='Viridis', showscale=True,
                colorbar=dict(title='H2 Projects'), line=dict(color='white',width=1)),
    hovertemplate='<b>%{text}</b><br>PGM: %{marker.size}<extra></extra>'))
fig.update_geos(scope='africa', showcountries=True, showcoastlines=True,
                lataxis_range=[-36,-20], lonaxis_range=[15,35],
                landcolor='#1c2128', oceancolor='#0d1117', bgcolor='#0d1117')
fig.update_layout(title='South Africa PGM Reserves & Hydrogen Project Footprint',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🗺 Map — PGM Reserves & Hydrogen Footprint', fig)

# --- 2.2 SANKEY: Strategy → Execution ---------------------------------------
labels = ['Strategies','PGM Advantage','Renewables','Coordination',
          'Investment','Skills','IMPLEMENTATION','Beneficiation']
src=[0,0,0,1,2,3,4,4,5,5,6]
tgt=[1,2,3,4,4,4,6,5,6,7,7]
val=[30,25,15,20,25,10,15,5,20,8,12]
fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=18,
              label=labels,
              color=['#58a6ff','#79c0ff','#3fb950','#d29922','#f85149','#a371f7','#2ea043','#ffa657']),
    link=dict(source=src, target=tgt, value=val,
              color='rgba(88,166,255,0.25)')))
fig.update_layout(title='Sankey — Flow: Strategy Aspiration → Execution Reality',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🔀 Sankey — Strategy → Execution Flow', fig)

# --- 2.3 SUNBURST: PGM value chain -----------------------------------------
sb = pd.DataFrame({
    'Level1':['Mining']*2+['Refining']*3+['PGM Products']*6+['Applications']*4,
    'Level2':['PGM Ore','Chromite','Smelting','Base Metal Ref','Precious Metal Ref',
              'Pt','Pd','Rh','Ru','Ir','Au','Catalysts','Fuel Cells','Electrolyzers','Auto-cat'],
    'Value':[45,20,30,18,25,40,35,18,8,5,10,40,30,22,35]})
fig = px.sunburst(sb, path=['Level1','Level2'], values='Value',
                  color='Value', color_continuous_scale='Turbo',
                  title='Sunburst — PGM Value Chain to Hydrogen Applications')
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('☀ Sunburst — PGM Value Chain', fig)

# --- 2.4 TREEMAP: Investment allocation ------------------------------------
fig = px.treemap(alloc, path=['Sector','Category'], values='Investment_Rbn',
                 color='Investment_Rbn', color_continuous_scale='Blues',
                 title='Treemap — R&B Investment Allocation by Sector')
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🌳 Treemap — Investment Allocation', fig)

# --- 2.5 3D SCATTER: Hydrogen projects -------------------------------------
fig = px.scatter_3d(df_proj, x='Capacity_MW', y='Investment_Rbn', z='Distance_Port_km',
                    color='Type', symbol='Province', size='Investment_Rbn',
                    color_discrete_sequence=px.colors.qualitative.Bold,
                    title='3D Scatter — Hydrogen Project Landscape')
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=600,
                  scene=dict(bgcolor='#0d1117'))
add('🔵 3D Scatter — Project Landscape', fig)

# --- 2.6 RADAR: Readiness vs target -----------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=current+[current[0]], theta=radar_cats+[radar_cats[0]],
    fill='toself', name='Current (2025)', line_color='#f85149'))
fig.add_trace(go.Scatterpolar(r=target+[target[0]], theta=radar_cats+[radar_cats[0]],
    fill='toself', name='Target (2030)', line_color='#3fb950'))
fig.update_layout(title='Radar — PGM-Hydrogen Economy Readiness',
                  polar=dict(bgcolor='#0d1117', radialaxis=dict(range=[0,10], gridcolor='#30363d')),
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🕸 Radar — Readiness vs Target', fig)

# --- 2.7 HEATMAP / IMSHOW: Correlation --------------------------------------
fig = px.imshow(C, x=metrics, y=metrics, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, text_auto='.2f',
                title='Heatmap — Execution Enabler Correlations')
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🔥 Heatmap — Enabler Correlations', fig)

# --- 2.8 CONTOUR (wafer-thickness analogy): Skills density ------------------
fig = go.Figure(go.Contour(z=Zc, x=np.linspace(0,10,50), y=np.linspace(0,10,50),
    colorscale='Turbo', contours=dict(showlabels=True),
    colorbar=dict(title='Skills Index')))
fig.update_layout(title='Contour — Skills Density Surface (wafer-thickness analogue)',
                  xaxis_title='Region X', yaxis_title='Region Y',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🗺 Contour — Skills Density Surface', fig)

# --- 2.9 3D SURFACE: Yield curve --------------------------------------------
fig = go.Figure(go.Surface(x=Xs, y=Ys, z=Zs, colorscale='Viridis',
                           colorbar=dict(title='Yield %')))
fig.update_layout(title='3D Yield Curve Surface — Investment vs Skills → Return',
                  scene=dict(xaxis_title='Investment', yaxis_title='Skills',
                             zaxis_title='ROI', bgcolor='#0d1117'),
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=600)
add('⛰ 3D Surface — ROI / Yield Curve', fig)

# --- 2.10 TIME SERIES SUBPLOT ----------------------------------------------
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Investment & Cumulative Projects','Skills & Jobs'),
                    vertical_spacing=0.15)
fig.add_trace(go.Bar(x=df_ts.Year, y=df_ts.Investment_Rbn, name='Investment (R bn)',
                     marker_color='#58a6ff'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_ts.Year, y=df_ts.Cum_Projects, name='Cum. Projects',
                         yaxis='y3', line=dict(color='#f85149', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_ts.Year, y=df_ts.Skills_Index, name='Skills Index',
                         line=dict(color='#3fb950', width=3)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_ts.Year, y=df_ts.Jobs_Created, name='Jobs',
                         line=dict(color='#d29922', dash='dot')), row=2, col=1)
fig.update_layout(title='Time Series — Investment, Skills, Projects, Jobs',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=650,
                  legend=dict(orientation='h', y=1.1))
add('📈 Time Series Subplots — Growth Trajectory', fig)

# --- 2.11 SCATTER PLOT MATRIX ----------------------------------------------
fig = px.scatter_matrix(df_proj, dimensions=['Capacity_MW','Investment_Rbn','Distance_Port_km'],
                        color='Type', title='Scatter Plot Matrix — Project Characteristics',
                        color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=600,
                  dragmode='select')
add('🔢 Scatter Matrix — Project Characteristics', fig)

# --- 2.12 FINANCIAL / ROI ---------------------------------------------------
fig = make_subplots(specs=[[{'secondary_y':True}]])
fig.add_trace(go.Scatter(x=df_fin.Date, y=df_fin.Price_Index,
                         name='Price Index', line=dict(color='#58a6ff', width=3),
                         fill='tozeroy', fillcolor='rgba(88,166,255,0.15)'))
fig.add_trace(go.Bar(x=df_fin.Date, y=df_fin.Revenue_Rbn, name='Revenue (R bn)',
                     marker_color='#3fb950', opacity=0.6), secondary_y=True)
fig.update_layout(title='Financial — Hydrogen Index vs Revenue (ROI proxy)',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=520)
add('💹 Financial — ROI / Revenue Index', fig)

# --- 2.13 COLLAR PLAN MAP --------------------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_collar.Easting, y=df_collar.Northing,
    mode='markers', marker=dict(size=df_collar.Depth_m/12,
    color=df_collar.PGM_ppm, colorscale='Hot', showscale=True,
    colorbar=dict(title='PGM ppm'), line=dict(color='white', width=0.5)),
    text=df_collar.PGM_ppm.round(2),
    hovertemplate='E=%{x:.0f}, N=%{y:.0f}<br>PGM=%{text} ppm<extra></extra>'))
fig.update_layout(title='Collar Plan Map — Drill Collars & PGM Grade',
                  xaxis_title='Easting (m)', yaxis_title='Northing (m)',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=600)
add('⛏ Collar Plan — PGM Drilling', fig)

# --- 2.14 NETWORK GRAPH: Stakeholder collaboration -------------------------
ex,ey = [],[]
for e in G.edges():
    x0,y0 = pos[e[0]]; x1,y1 = pos[e[1]]
    ex += [x0,x1,None]; ey += [y0,y1,None]
nx_ = [pos[n][0] for n in G.nodes()]
ny_ = [pos[n][1] for n in G.nodes()]
deg = [G.degree(n) for n in G.nodes()]
fig = go.Figure()
fig.add_trace(go.Scatter(x=ex, y=ey, mode='lines',
                         line=dict(color='rgba(88,166,255,0.35)', width=1.5),
                         hoverinfo='none'))
fig.add_trace(go.Scatter(x=nx_, y=ny_, mode='markers+text',
                         text=list(G.nodes()), textposition='top center',
                         marker=dict(size=[12+d*4 for d in deg], color=deg,
                                     colorscale='Plasma', showscale=True,
                                     colorbar=dict(title='Degree'),
                                     line=dict(color='white',width=1)),
                         hoverinfo='text'))
fig.update_layout(title='Network Graph — Government · Industry · Academia · Finance',
                  xaxis=dict(showgrid=False, zeroline=False, visible=False),
                  yaxis=dict(showgrid=False, zeroline=False, visible=False),
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=620)
add('🕸 Network Graph — Stakeholder Collaboration', fig)

# --- 2.15 ANIMATED SCATTER (basic + animation) -----------------------------
anim_rows = []
for yr in range(2020,2031):
    for _,r in df_proj.sample(20, random_state=yr).iterrows():
        anim_rows.append({'Year':yr,'Capacity_MW':r.Capacity_MW,
                          'Investment_Rbn':r.Investment_Rbn*(1+0.04*(yr-2020)),
                          'Type':r.Type})
df_anim = pd.DataFrame(anim_rows)
fig = px.scatter(df_anim, x='Capacity_MW', y='Investment_Rbn', color='Type',
                 animation_frame='Year', size='Investment_Rbn',
                 range_x=[0,520], range_y=[0,45],
                 title='Animated Scatter — Project Build-out 2020 → 2030',
                 color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(paper_bgcolor='#161b22', font_color='#e6edf3', height=560)
add('🎬 Animated Scatter — Build-out 2020–2030', fig)

# --- 2.16 BASIC + STATISTICAL SUBPLOTS (bars, box, histogram, pie) --------
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Provincial PGM Reserves','Investment by Sector',
                                    'Project Size Distribution','H2 Project Share'),
                    specs=[[{'type':'bar'},{'type':'bar'}],
                           [{'type':'histogram'},{'type':'pie'}]])
fig.add_trace(go.Bar(x=dfp.Province, y=dfp.PGM_Reserves_Moz,
                     marker_color='#58a6ff', name='PGM'), row=1, col=1)
sec = alloc.groupby('Sector')['Investment_Rbn'].sum().reset_index()
fig.add_trace(go.Bar(x=sec.Sector, y=sec.Investment_Rbn,
                     marker_color='#3fb950', name='Invest'), row=1, col=2)
fig.add_trace(go.Histogram(x=df_proj.Capacity_MW, nbinsx=15,
                           marker_color='#d29922', name='Cap'), row=2, col=1)
fig.add_trace(go.Pie(labels=df_proj.Type.value_counts().index,
                     values=df_proj.Type.value_counts().values,
                     hole=0.4, name='Type'), row=2, col=2)
fig.update_layout(title='Statistical Overview — PGM · Investment · Projects',
                  paper_bgcolor='#161b22', font_color='#e6edf3', height=700,
                  showlegend=False)
add('📊 Statistical Overview — Basics', fig)

# --- 2.17 CUSTOM CONTROL: Time series with scenario buttons ---------------
base    = df_ts.Investment_Rbn
accel   = base*1.45
delay   = base*0.72
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_ts.Year, y=base, name='Base',
                         line=dict(color='#58a6ff', width=3)))
for name, y, c in [('Accelerated', accel,'#3fb950'), ('Delayed', delay,'#f85149')]:
    fig.add_trace(go.Scatter(x=df_ts.Year, y=y, name=name, visible=False,
                             line=dict(color=c, width=3)))
fig.update_layout(
    title='Interactive Time Series — Investment Scenarios (use buttons)',
    paper_bgcolor='#161b22', font_color='#e6edf3', height=520,
    updatemenus=[dict(type='buttons', direction='right', x=0.5, y=1.18,
        xanchor='center', showactive=True, buttons=[
            dict(label='Base', method='update',
                 args=[{'visible':[True,False,False]}]),
            dict(label='Accelerated', method='update',
                 args=[{'visible':[False,True,False]}]),
            dict(label='Delayed', method='update',
                 args=[{'visible':[False,False,True]}]),
        ])])
add('🎛 Interactive — Investment Scenarios', fig)

# =============================================================================
# 3. EXPORT — combined HTML dashboard
# =============================================================================
html = ["<html><head><meta charset='utf-8'>",
        "<title>MISTRA 2026 — PGM-Hydrogen Dashboard</title>",
        "<style>body{font-family:Arial;background:#0d1117;color:#e6edf3;padding:20px}",
        "h1{color:#58a6ff}h2{color:#79c0ff;border-bottom:1px solid #30363d;padding-bottom:6px}",
        ".fig{margin:24px 0;background:#161b22;padding:12px;border-radius:10px}",
        ".note{color:#8b949e;font-size:13px}</style></head><body>",
        "<h1>MISTRA 2026 Annual PGM Roundtable</h1>",
        "<p class='note'>From strategic aspiration to execution reality — "
        "South Africa's PGM-powered hydrogen economy · Synthetic demonstration data</p>"]

first = True
for name, f in FIGS.items():
    html.append(f"<div class='fig'><h2>{name}</h2>")
    html.append(to_html(f, full_html=False, include_plotlyjs='cdn' if first else False))
    html.append("</div>")
    first = False
html.append("</body></html>")

with open("mistra_pgm_h2_dashboard.html", "w", encoding="utf-8") as fh:
    fh.write("\n".join(html))

print("✅ Dashboard saved: mistra_pgm_h2_dashboard.html")
print(f"✅ Total figures: {len(FIGS)}")
print("   → Open in a browser. All charts are interactive.")
for k in FIGS: print("   •", k)

✅ Dashboard saved: mistra_pgm_h2_dashboard.html
✅ Total figures: 17
   → Open in a browser. All charts are interactive.
   • 🗺 Map — PGM Reserves & Hydrogen Footprint
   • 🔀 Sankey — Strategy → Execution Flow
   • ☀ Sunburst — PGM Value Chain
   • 🌳 Treemap — Investment Allocation
   • 🔵 3D Scatter — Project Landscape
   • 🕸 Radar — Readiness vs Target
   • 🔥 Heatmap — Enabler Correlations
   • 🗺 Contour — Skills Density Surface
   • ⛰ 3D Surface — ROI / Yield Curve
   • 📈 Time Series Subplots — Growth Trajectory
   • 🔢 Scatter Matrix — Project Characteristics
   • 💹 Financial — ROI / Revenue Index
   • ⛏ Collar Plan — PGM Drilling
   • 🕸 Network Graph — Stakeholder Collaboration
   • 🎬 Animated Scatter — Build-out 2020–2030
   • 📊 Statistical Overview — Basics
   • 🎛 Interactive — Investment Scenarios


In [3]:

# =============================================================================
# MISTRA 2026 ANNUAL PGM ROUNDTABLE — South Africa's PGM-Powered Hydrogen Economy
# "From Strategic Aspirations to Execution Reality"
# -----------------------------------------------------------------------------
# Single-cell Python project demonstrating:
#   synthetic data | Plotly | Matplotlib | NetworkX | scikit-learn | PyTorch
#   SQL (sqlite) | Jupyter widgets | animation | maps | sankey | sunburst
#   heatmaps | 3D | contour | treemap | scatter matrix | radar | time series
#   ROI / financial charts | bioinformatics | HMI/SCADA-style dashboards
#   LLM/MCP-ready payload builders | Kubernetes/GPU hints (as metadata)
# =============================================================================

# ----------------------------- 0. IMPORTS -----------------------------------
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt, networkx as nx, sqlite3, warnings, json, io
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings("ignore")
np.random.seed(42)

# ----------------------- 1. SYNTHETIC DATA LAYER ----------------------------
PROVINCES = ["Gauteng","Limpopo","North West","Mpumalanga","Free State",
             "KwaZulu-Natal","Eastern Cape","Western Cape","Northern Cape"]
PGM_MINES  = ["Anglo American Platinum","Sibanye-Stillwater","Impala Platinum",
              "Northam Platinum","Royal Bafokeng","Valterra Platinum"]

data = {
    "province": PROVINCES,
    "pgm_reserve_koz": np.random.randint(500, 12000, len(PROVINCES)),
    "hydrogen_capacity_mw": np.random.randint(20, 900, len(PROVINCES)),
    "investment_zar_bn": np.round(np.random.uniform(0.5, 45, len(PROVINCES)),2),
    "jobs_2030": np.random.randint(800, 22000, len(PROVINCES)),
    "skills_index": np.round(np.random.uniform(0.3, 0.95, len(PROVINCES)),2),
    "lat": [-26.2,-23.9,-25.8,-25.5,-29.1,-29.0,-32.3,-33.9,-29.0],
    "lon": [28.0, 29.4, 25.6, 30.9, 26.2, 30.5, 26.9, 20.9, 21.5],
}
df = pd.DataFrame(data)

# time series (2024–2035) — capacity ramp and ROI
years = np.arange(2024, 2036)
ts = pd.DataFrame({
    "year": years,
    "hydrogen_mw": 150 * np.exp(0.28*(years-2024)) + np.random.normal(0,20,len(years)),
    "capex_bn":   12 + 3.2*(years-2024) + np.random.normal(0,0.8,len(years)),
    "revenue_bn": 0.2 + 1.4*np.exp(0.22*(years-2024)),
    "roi_pct":    2 + 1.9*(years-2024) + np.random.normal(0,0.6,len(years)),
})

# stack/stakeholder network
stake = ["Government","MISTRA","Creamer Media","PGM Miners","Hydrogen OEMs",
         "DFIs/Banks","Universities/TVET","Emerging SMEs","Fuel-Cell Users","Communities"]
G = nx.Graph()
edges = [("Government","MISTRA"),("Government","DFIs/Banks"),("Government","PGM Miners"),
         ("MISTRA","Creamer Media"),("MISTRA","Universities/TVET"),("Creamer Media","PGM Miners"),
         ("PGM Miners","Hydrogen OEMs"),("PGM Miners","Emerging SMEs"),("DFIs/Banks","Hydrogen OEMs"),
         ("Hydrogen OEMs","Fuel-Cell Users"),("Universities/TVET","Emerging SMEs"),
         ("Emerging SMEs","Fuel-Cell Users"),("Communities","Government"),("Communities","PGM Miners")]
G.add_edges_from(edges)
pos = nx.spring_layout(G, seed=7)

# ------------------- 2. BASIC + STATISTICAL CHARTS --------------------------
fig1 = px.bar(df, x="province", y="pgm_reserve_koz", color="investment_zar_bn",
              title="PGM Reserves vs Investment by Province",
              color_continuous_scale="Viridis", template="plotly_white")
fig1.update_layout(xaxis_tickangle=-35); fig1.show()

fig2 = px.scatter(df, x="hydrogen_capacity_mw", y="jobs_2030",
                  size="investment_zar_bn", color="skills_index",
                  hover_name="province", trendline="ols",
                  title="Capacity → Jobs (size=Investment, color=Skills)",
                  template="plotly_white")
fig2.show()

# --------------------------- 3. FINANCIAL / ROI -----------------------------
fig3 = make_subplots(rows=1, cols=2, subplot_titles=("Capex vs Revenue","ROI %"))
fig3.add_trace(go.Bar(x=ts.year, y=ts.capex_bn,   name="Capex (ZAR bn)", marker_color="#c0392b"),1,1)
fig3.add_trace(go.Bar(x=ts.year, y=ts.revenue_bn,name="Revenue (ZAR bn)",marker_color="#27ae60"),1,1)
fig3.add_trace(go.Scatter(x=ts.year,y=ts.roi_pct,mode="lines+markers",
                          line=dict(color="#2980b9",width=3)),1,2)
fig3.update_layout(title="Hydrogen Economy Financial Projection", barmode="group", template="plotly_white")
fig3.show()

# ------------------------------ 4. MAP --------------------------------------
fig4 = px.scatter_geo(df, lat="lat", lon="lon", size="hydrogen_capacity_mw",
                      color="province", hover_name="province",
                      scope="africa", title="South Africa Hydrogen Hubs (GIS)",
                      template="plotly_white")
fig4.update_geos(fitbounds="locations", showcountries=True); fig4.show()

# ------------------------- 5. 3D SCATTER + SURFACE --------------------------
fig5 = px.scatter_3d(df, x="pgm_reserve_koz", y="hydrogen_capacity_mw",
                     z="investment_zar_bn", color="province", size="jobs_2030",
                     title="3-D PGM–H₂–Investment Space", template="plotly_white")
fig5.show()

xx, yy = np.meshgrid(np.linspace(0,1,50), np.linspace(0,1,50))
zz = np.exp(-((xx-0.5)**2 + (yy-0.5)**2)/0.08)
fig6 = go.Figure(data=[go.Surface(z=zz, x=xx, y=yy, colorscale="Turbo")])
fig6.update_layout(title="3-D Yield Curve Surface (H₂ Plant Efficiency)",
                   scene=dict(xaxis_title="Load", yaxis_title="Temp", zaxis_title="Yield"))
fig6.show()

# -------------------------- 6. SANKEY / SUNBURST ----------------------------
fig7 = go.Figure(go.Sankey(
    node=dict(label=["PGM Ore","Refining","Catalysts","Electrolysers",
                     "H₂ Production","Fuel Cells","Export","Domestic Use","Jobs"],
              pad=18, thickness=22),
    link=dict(source=[0,1,1,2,3,4,5,5,4],
              target=[1,2,3,3,4,5,6,7,8],
              value =[100,45,55,50,80,60,30,25,20])))
fig7.update_layout(title="PGM → Hydrogen Value Chain (Sankey)"); fig7.show()

fig8 = px.sunburst(df, path=["province"], values="investment_zar_bn",
                   color="skills_index", title="Investment Sunburst",
                   color_continuous_scale="RdYlGn")
fig8.show()

# --------------------------- 7. HEATMAP / TREEMAP ---------------------------
pivot = df.pivot_table(index="province", values=["skills_index","investment_zar_bn"])
fig9 = px.imshow(pivot.T, text_auto=".2f", aspect="auto",
                 color_continuous_scale="YlOrRd", title="Provincial Indicators Heatmap")
fig9.show()

fig10 = px.treemap(df, path=["province"], values="pgm_reserve_koz",
                   color="investment_zar_bn", title="PGM Reserve Treemap")
fig10.show()

# ----------------------- 8. TIME SERIES SUBPLOTS ----------------------------
fig11 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                      subplot_titles=("H₂ Capacity MW","ROI %"))
fig11.add_trace(go.Scatter(x=ts.year,y=ts.hydrogen_mw,fill="tozeroy"),1,1)
fig11.add_trace(go.Scatter(x=ts.year,y=ts.roi_pct,mode="lines+markers"),2,1)
fig11.update_layout(title="2024–2035 Ramp-up",template="plotly_white"); fig11.show()

# ------------------------------ 9. RADAR ------------------------------------
cats = ["Policy","Capital","Skills","Tech","Partners","Markets"]
fig12 = go.Figure()
for prov,row in df.head(4).iterrows():
    vals = [row.skills_index, row.investment_zar_bn/50, row.skills_index,
            row.hydrogen_capacity_mw/900, row.jobs_2030/22000, row.skills_index]
    fig12.add_trace(go.Scatterpolar(r=vals, theta=cats, fill="toself", name=row.province))
fig12.update_layout(title="Readiness Radar (4 provinces)", polar=dict(radialaxis=dict(range=[0,1])))
fig12.show()

# ------------------------ 10. NETWORK GRAPH (STAKEHOLDERS) ------------------
edge_x, edge_y = [], []
for e in G.edges():
    edge_x += [pos[e[0]][0], pos[e[1]][0], None]
    edge_y += [pos[e[0]][1], pos[e[1]][1], None]
node_x = [pos[n][0] for n in G.nodes]; node_y = [pos[n][1] for n in G.nodes]
fig13 = go.Figure()
fig13.add_trace(go.Scatter(x=edge_x,y=edge_y,mode="lines",line=dict(color="#888",width=1.5)))
fig13.add_trace(go.Scatter(x=node_x,y=node_y,mode="markers+text",text=list(G.nodes),
                           textposition="top center",marker=dict(size=22,color="steelblue")))
fig13.update_layout(title="Stakeholder Network (Government–Industry–Academia)",
                    showlegend=False); fig13.show()

# ------------------------ 11. SCATTER MATRIX --------------------------------
fig14 = px.scatter_matrix(df, dimensions=["pgm_reserve_koz","hydrogen_capacity_mw",
                                          "investment_zar_bn","jobs_2030","skills_index"],
                          color="province", title="Scatter Plot Matrix")
fig14.update_traces(diagonal_visible=False); fig14.show()

# ------------------------ 12. CONTOUR + IMF --------------------------------
fig15 = go.Figure(go.Contour(z=zz, x=xx[0], y=yy[:,0],
                             colorscale="Portland", contours=dict(showlabels=True)))
fig15.update_layout(title="Wafer-Thickness Style Contour (H₂ Plant Yield)")
fig15.show()

# ------------------- 13. ML: PCA + Regression (execution gap) ---------------
X = df[["pgm_reserve_koz","hydrogen_capacity_mw","investment_zar_bn","skills_index"]].values
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit_transform(Xs)
df["PC1"], df["PC2"] = pca[:,0], pca[:,1]
fig16 = px.scatter(df, x="PC1", y="PC2", color="province", size="jobs_2030",
                   title="AI/ML: PCA of Provincial Execution Readiness")
fig16.show()

lr = LinearRegression().fit(X[:, [1,2,3]], df.jobs_2030)
print("Regression coefficients (capacity, investment, skills):", np.round(lr.coef_,2))
print("R² on training:", round(lr.score(X[:, [1,2,3]], df.jobs_2030),3))

# --------------------- 14. PYTORCH (tiny forecasting net) -------------------
import torch, torch.nn as nn
torch.manual_seed(0)
Xt = torch.tensor(ts[["year"]].values - 2024, dtype=torch.float32)
yt = torch.tensor(ts[["hydrogen_mw"]].values/1000.0, dtype=torch.float32)
net = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,1))
opt = torch.optim.Adam(net.parameters(), lr=0.05); lossf = nn.MSELoss()
for _ in range(800):
    opt.zero_grad(); loss = lossf(net(Xt), yt); loss.backward(); opt.step()
with torch.no_grad():
    pred = net(Xt).numpy().ravel()*1000
fig17 = go.Figure()
fig17.add_trace(go.Scatter(x=ts.year,y=ts.hydrogen_mw,mode="markers",name="Actual"))
fig17.add_trace(go.Scatter(x=ts.year,y=pred,mode="lines",name="PyTorch Fit"))
fig17.update_layout(title="PyTorch MLP Fit — H₂ Capacity Ramp",template="plotly_white")
fig17.show()

# --------------------------- 15. SQL LAYER ---------------------------------
con = sqlite3.connect(":memory:"); df.to_sql("hubs", con, index=False)
q = "SELECT province, SUM(investment_zar_bn) inv, MAX(hydrogen_capacity_mw) cap FROM hubs GROUP BY province ORDER BY inv DESC"
print("\nTop provinces by investment (SQL):\n", pd.read_sql(q, con).head())

# --------------------------- 16. BIOINFORMATICS demo -----------------------
bio = pd.DataFrame({"gene":["H2aseA","H2aseB","PGMred1","CAT1","SKL1"],
                    "expression":np.random.uniform(0.2,1,5)})
fig18 = px.bar(bio, x="gene", y="expression", title="Bioinformatics-style Expression (H₂ pathway genes)",
               color="expression", color_continuous_scale="Teal")
fig18.show()

# --------------------------- 17. HMI / SCADA gauge -------------------------
fig19 = make_subplots(rows=1, cols=3, specs=[[{"type":"indicator"}]*3],
                      subplot_titles=("Electrolyser Load","Plant Pressure","H₂ Purity"))
for i,(val,lbl) in enumerate([(78,"%"),(4.2,"bar"),(99.9,"%")]):
    fig19.add_trace(go.Indicator(mode="gauge+number", value=val,
        gauge=dict(axis=dict(range=[0,100]) if i!=1 else dict(range=[0,10])),
        title=dict(text=lbl)),1,i+1)
fig19.update_layout(title="HMI/SCADA Plant Dashboard"); fig19.show()

# --------------------------- 18. ANIMATION ---------------------------------
anim = px.scatter(df, x="hydrogen_capacity_mw", y="jobs_2030",
                  size="investment_zar_bn", color="province",
                  animation_frame="skills_index", range_x=[0,1000], range_y=[0,25000],
                  title="Animated Readiness (skills_index as frame)")
anim.show()

# --------------------------- 19. JUPYTER WIDGETS ---------------------------
slider = widgets.FloatSlider(value=5, min=0, max=50, step=0.5, description="ZAR bn:")
out = widgets.Output()
def _show(change):
    with out:
        out.clear_output()
        sub = df[df.investment_zar_bn >= change["new"]]
        display(px.bar(sub, x="province", y="jobs_2030",
                       title=f"Jobs (investment ≥ {change['new']} bn)"))
slider.observe(_show, names="value"); _show({"new":5})
display(slider, out)

# --------------------------- 20. LLM / MCP PAYLOAD -------------------------
mcp_payload = {
    "tool":"mistra_pgm_hydrogen_roundtable",
    "server":"creamermedia-mcp",
    "context":{
        "event":"2026 MISTRA Annual PGM Roundtable",
        "pillars":["Policy→Implementation","Investment","Ecosystem","Skills","Collaboration","Industrialisation"],
        "gpu_hint":"nvidia-a100 (analytics)","k8s":"mistra-analytics-ns",
        "charts":["bar","scatter","sankey","sunburst","heatmap","treemap",
                  "radar","network","3D","contour","gauge","animation","PCA","PyTorch"]
    },
    "data_summary": df[["province","investment_zar_bn","jobs_2030"]].to_dict("records")
}
print("\nMCP/LLM payload ready:", json.dumps(mcp_payload["context"]["pillars"]))

# --------------------------- 21. DASH ENTERPRISE-style HTML ---------------
html = """<div style='font-family:Arial;padding:12px;background:#0b1e3a;color:#fff'>
<h2>MISTRA 2026 · PGM-Powered Hydrogen Economy Dashboard</h2>
<p>Execution reality: capital, skills, ecosystem & collaboration in one view.</p>
</div>"""
from IPython.display import HTML as HTML_
display(HTML_(html))

print("\n✅ All modules executed: charts, maps, ML, PyTorch, SQL, widgets, animation,")
print("   HMI/SCADA, bioinformatics, MCP/LLM payload — ready for Dash/Plotly Studio export.")

Regression coefficients (capacity, investment, skills): [ 5.74000e+00  4.85210e+02 -7.33008e+03]
R² on training: 0.424



Top provinces by investment (SQL):
         province    inv  cap
0        Gauteng  43.66  478
1        Limpopo  37.54  107
2   Eastern Cape  23.85  150
3   Western Cape  19.72  681
4  KwaZulu-Natal  14.04  683


FloatSlider(value=5.0, description='ZAR bn:', max=50.0, step=0.5)

Output()


MCP/LLM payload ready: ["Policy\u2192Implementation", "Investment", "Ecosystem", "Skills", "Collaboration", "Industrialisation"]



✅ All modules executed: charts, maps, ML, PyTorch, SQL, widgets, animation,
   HMI/SCADA, bioinformatics, MCP/LLM payload — ready for Dash/Plotly Studio export.


In [4]:

# =============================================================================
# MISTRA 2026 ANNUAL PGM ROUNDTABLE
# "A transition from strategic aspirations to execution reality:
#  Accelerating South Africa's PGM-Powered Hydrogen Economy"
# One-cell analytic / visualization demo (synthetic data)
# =============================================================================
#   MCP + LLM + Figma MCP • Plotly Studio / Dash Enterprise-style dashboard
#   Basic + statistical + financial + ROI • Radar • Network • Sankey • Sunburst
#   Treemap • Heatmap • Scatter-matrix • Geo map • 3D scatter • 3D yield surface
#   Wafer contour • Collar plan map • Time-series subplot • Animation
#   AI/ML (sklearn + PyTorch) • Bioinformatics • Image data (imshow)
#   Jupyter widgets + custom controls • HMI / SCADA • SQL mock • Kubernetes
#   amCharts + SofaAir-style HTML • Google-Studio-style KPI cards
# =============================================================================

import warnings, json, random, math, textwrap
warnings.filterwarnings("ignore")
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

try:    import networkx as nx
except: nx = None
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
except: SKLEARN_OK = False
try:
    import torch, torch.nn as nn
    TORCH_OK = True
except: TORCH_OK = False
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    IPY_OK = True
except: IPY_OK = False

np.random.seed(42); random.seed(42)
DARK = "plotly_dark"

def safe_html(html):
    if IPY_OK:
        display(HTML(html))
    else:
        print(html)

def safe_display(obj):
    if IPY_OK:
        display(obj)
    else:
        print(obj)

# =============================================================================
# 1) SYNTHETIC DATA
# =============================================================================
YEARS = list(range(2020, 2031))
df = pd.DataFrame({
    "Year": YEARS,
    "PGM_Production_Moz":          np.array([4.6,4.8,4.5,4.4,4.3,4.2,4.4,4.6,4.8,5.0,5.2]),
    "Hydrogen_Investment_USDbn":   np.array([0.1,0.3,0.6,1.2,2.1,3.4,5.0,7.2,9.8,12.5,15.0]),
    "PGM_Revenue_ZARbn":           np.array([120,135,128,150,165,175,190,205,220,235,250]),
    "Jobs_Thousands":              np.array([12,15,18,22,28,35,44,55,68,82,98]),
    "CO2_Reduction_MT":            np.array([0.2,0.4,0.7,1.1,1.6,2.3,3.1,4.0,5.1,6.4,7.8]),
})

skills_df = pd.DataFrame({
    "Skill": ["Chemical Eng.","Materials Sci.","Electrical Eng.","Data Science",
              "Metallurgy","Fuel Cell Tech.","Project Finance","Policy Analysts",
              "HSE Specialists","Grid Engineers"],
    "Current_Supply":        [1800, 900, 2500, 1200, 1400, 320, 600, 450, 800, 1100],
    "Projected_Demand_2030": [3200, 2000, 3900, 2600, 2400, 900, 1400, 900, 1500, 2000],
})
skills_df["Gap"] = skills_df["Projected_Demand_2030"] - skills_df["Current_Supply"]

# =============================================================================
# 2) MCP SERVER (mock)
# =============================================================================
class MCPServer:
    """Mock Model Context Protocol server — tool registry & JSON dispatch."""
    def __init__(self, name):
        self.name, self.tools, self.call_log = name, {}, []
    def register(self, name, fn, desc=""):
        self.tools[name] = {"fn": fn, "desc": desc}
    def list_tools(self):
        return [{"name": k, "description": v["desc"]} for k, v in self.tools.items()]
    def call(self, name, **kw):
        self.call_log.append((datetime.now().isoformat(timespec="seconds"), name, kw))
        if name not in self.tools: return {"error": f"Unknown tool: {name}"}
        return self.tools[name]["fn"](**kw)

mcp = MCPServer("MISTRA-PGM-Hydrogen-MCP")
mcp.register("get_pgm_production",
             lambda year: float(df.loc[df.Year==year,"PGM_Production_Moz"].iloc[0]),
             "PGM production (Moz) for a given year")
mcp.register("get_h2_investment",
             lambda year: float(df.loc[df.Year==year,"Hydrogen_Investment_USDbn"].iloc[0]),
             "Hydrogen investment (USD bn) for a given year")
mcp.register("skills_gap",
             lambda skill: float(skills_df.loc[skills_df.Skill==skill,"Gap"].iloc[0]),
             "2030 skills gap for a named occupation")
mcp.register("roi_calc",
             lambda capex, cashflows, rate=0.10:
                 float(np.sum([cf/(1+rate)**i for i, cf in enumerate(cashflows, 1)]) - capex),
             "NPV-style ROI over cashflows")

# =============================================================================
# 3) LLM (mock) — grounded in MCP tools
# =============================================================================
class MockLLM:
    def __init__(self, mcp, df, skills_df):
        self.mcp, self.df, self.skills = mcp, df, skills_df
    def respond(self, prompt: str) -> str:
        p = prompt.lower()
        if "investment" in p:
            return (f"Total projected H₂ investment 2020–2030 ≈ "
                    f"USD {self.df.Hydrogen_Investment_USDbn.sum():.1f} bn. "
                    f"Biggest single-year jump is 2029→2030 "
                    f"(${self.mcp.call('get_h2_investment', year=2029):.1f} → "
                    f"${self.mcp.call('get_h2_investment', year=2030):.1f} bn).")
        if "skill" in p or "workforce" in p:
            top = self.skills.sort_values("Gap", ascending=False).iloc[0]
            return (f"Largest skills gap is **{top.Skill}** "
                    f"({int(top.Gap)} additional specialists needed by 2030).")
        if "roi" in p or "npv" in p:
            npv = self.mcp.call("roi_calc", capex=450,
                                cashflows=[60,85,110,130,145,150,155,160], rate=0.10)
            return (f"Typical PGM-H₂ project NPV @ 10% ≈ USD {npv:.1f} m, "
                    f"payback ≈ 6–9 years.")
        if "sankey" in p or "value chain" in p:
            return "Value chain: PGM ore → refining → catalysts & electrolysers → green H₂ → industry/export."
        return "Question logged; MCP tools available for quantitative checks."

llm = MockLLM(mcp, df, skills_df)

# =============================================================================
# 4) FIGMA MCP (mock) — design tokens + frame layout
# =============================================================================
class FigmaMCP:
    def __init__(self):
        self.tokens = {"primary":"#0B6E4F","secondary":"#FFB703","bg":"#0E1116",
                       "text":"#E6EDF3","accent":"#F25C54","grid":"#1F2937"}
    def get_design_tokens(self): return self.tokens
    def get_frame(self, name):
        return {"frame":name,"size":[1440,900],
                "layers":["header","kpi_row","charts_grid","footer"]}
figma = FigmaMCP()

# =============================================================================
# 5) HTML / Google-Studio-style KPI header
# =============================================================================
kpi = df[df.Year==2026].iloc[0]
safe_html(f"""
<div style="background:{figma.tokens['bg']};color:{figma.tokens['text']};
            padding:16px;border-radius:12px;font-family:Inter,system-ui,sans-serif">
  <h2 style="margin:0;color:{figma.tokens['secondary']}">
    MISTRA 2026 · PGM-Powered Hydrogen Economy</h2>
  <p style="margin:4px 0 14px 0;color:#94a3b8">
    Strategic aspiration → execution reality · synthetic demo dashboard</p>
  <div style="display:flex;gap:16px;flex-wrap:wrap">
    <div style="background:{figma.tokens['grid']};padding:12px;border-radius:8px;min-width:180px">
      <div style="font-size:12px;color:#94a3b8">PGM production (Moz)</div>
      <div style="font-size:26px;color:{figma.tokens['primary']}">{kpi.PGM_Production_Moz}</div></div>
    <div style="background:{figma.tokens['grid']};padding:12px;border-radius:8px;min-width:180px">
      <div style="font-size:12px;color:#94a3b8">H₂ investment (USD bn)</div>
      <div style="font-size:26px;color:{figma.tokens['secondary']}">{kpi.Hydrogen_Investment_USDbn}</div></div>
    <div style="background:{figma.tokens['grid']};padding:12px;border-radius:8px;min-width:180px">
      <div style="font-size:12px;color:#94a3b8">Jobs (thousands)</div>
      <div style="font-size:26px;color:{figma.tokens['accent']}">{kpi.Jobs_Thousands}</div></div>
    <div style="background:{figma.tokens['grid']};padding:12px;border-radius:8px;min-width:180px">
      <div style="font-size:12px;color:#94a3b8">CO₂ avoided (Mt)</div>
      <div style="font-size:26px;color:#2a9d8f">{kpi.CO2_Reduction_MT}</div></div>
  </div>
</div>
""")

# =============================================================================
# 6) BASIC CHART — dual-axis bar + line
# =============================================================================
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=df.Year, y=df.PGM_Production_Moz, name="PGM (Moz)",
                     marker_color="#2a9d8f"), secondary_y=False)
fig.add_trace(go.Scatter(x=df.Year, y=df.PGM_Revenue_ZARbn, name="Revenue (ZAR bn)",
                         mode="lines+markers", line=dict(color="#ffb703")), secondary_y=True)
fig.update_layout(title="Basic Chart — PGM Production vs Revenue",
                  template=DARK, height=400)
fig.show()

# =============================================================================
# 7) TIME-SERIES SUBPLOT (2×2)
# =============================================================================
fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    subplot_titles=("PGM production","H₂ investment",
                                    "Jobs","CO₂ reduction"))
fig.add_trace(go.Scatter(x=df.Year, y=df.PGM_Production_Moz,
                         line=dict(color="#2a9d8f"), name="PGM"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.Year, y=df.Hydrogen_Investment_USDbn,
                         line=dict(color="#ffb703"), name="H₂"), row=1, col=2)
fig.add_trace(go.Scatter(x=df.Year, y=df.Jobs_Thousands,
                         line=dict(color="#f4a261"), name="Jobs"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.Year, y=df.CO2_Reduction_MT,
                         line=dict(color="#e76f51"), name="CO₂"), row=2, col=2)
fig.update_layout(height=560, showlegend=False,
                  title="Time-Series Subplot Dashboard", template=DARK)
fig.show()

# =============================================================================
# 8) STATISTICAL CHARTS — histogram + box
# =============================================================================
samples = np.random.normal(loc=df.Hydrogen_Investment_USDbn.mean(), scale=2.5, size=800)
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Simulated H₂ investment distribution",
                                    "Jobs box-plot"))
fig.add_trace(go.Histogram(x=samples, nbinsx=35,
                           marker_color="#457b9d", name="samples"), row=1, col=1)
fig.add_trace(go.Box(y=df.Jobs_Thousands, name="Jobs (k)",
                     marker_color="#e76f51"), row=1, col=2)
fig.update_layout(template=DARK, height=380, title="Statistical Charts")
fig.show()

# =============================================================================
# 9) FINANCIAL / ROI — cumulative cashflow + waterfall
# =============================================================================
capex = 450.0
cashflows = [60, 85, 110, 130, 145, 150, 155, 160]
disc = 0.10
npv = sum(cf/(1+disc)**(i+1) for i, cf in enumerate(cashflows)) - capex
cum = np.cumsum(cashflows) - capex
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(f"Cumulative cashflow (NPV ≈ {npv:.1f} M USD)",
                                    "Waterfall — value chain"))
fig.add_trace(go.Scatter(x=list(range(1, 9)), y=cum, mode="lines+markers",
                         fill="tozeroy", line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Waterfall(
    x=["Capex","Opex savings","Revenue","Tax","Net"],
    y=[-capex, 40, 500, -120, -80],
    measure=["relative","relative","relative","relative","total"],
    textposition="outside"), row=1, col=2)
fig.update_layout(template=DARK, height=420, showlegend=False,
                  title="Financial & ROI Analysis")
fig.show()

# =============================================================================
# 10) RADAR — readiness vs target
# =============================================================================
cats    = ["Policy","Investment","Skills","Infrastructure","Innovation","Collaboration"]
current = [6, 4, 3, 5, 5, 4]
target  = [9, 8, 8, 9, 8, 9]
fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=current, theta=cats, fill="toself",
                              name="2024 readiness", line_color="#ffb703"))
fig.add_trace(go.Scatterpolar(r=target, theta=cats, fill="toself",
                              name="2030 target", line_color="#2a9d8f"))
fig.update_layout(template=DARK, polar=dict(radialaxis=dict(range=[0, 10])),
                  title="Radar — Strategic Aspiration vs Execution Target")
fig.show()

# =============================================================================
# 11) NETWORK GRAPH — stakeholder collaboration
# =============================================================================
if nx is not None:
    edges = [
        ("Government (DMRE)","IDC"),("Government (DMRE)","Minerals Council"),
        ("Government (DMRE)","CSIR"),("MISTRA","Government (DMRE)"),
        ("MISTRA","Universities"),("MISTRA","Minerals Council"),
        ("Anglo American Platinum","Sibanye-Stillwater"),
        ("Impala Platinum","Anglo American Platinum"),
        ("Sasol","Eskom"),("Sasol","IDC"),("CSIR","Universities"),
        ("Universities","Fuel Cell OEMs"),("Fuel Cell OEMs","Private Investors"),
        ("Hydrogen Valley","Government (DMRE)"),("Hydrogen Valley","Sasol"),
        ("Emerging Miners","Minerals Council"),
        ("Development Bank of SA","IDC"),("Development Bank of SA","Private Investors"),
        ("Private Investors","Hydrogen Valley"),
        ("Sibanye-Stillwater","Emerging Miners"),("Impala Platinum","CSIR"),
        ("Eskom","IDC"),
    ]
    G = nx.Graph(); G.add_edges_from(edges)
    pos = nx.spring_layout(G, seed=42)
    ex, ey = [], []
    for a, b in G.edges():
        x0, y0 = pos[a]; x1, y1 = pos[b]
        ex += [x0, x1, None]; ey += [y0, y1, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
                             line=dict(width=1, color="#475569"), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=[pos[n][0] for n in G.nodes()],
        y=[pos[n][1] for n in G.nodes()],
        mode="markers+text", text=list(G.nodes()), textposition="top center",
        marker=dict(size=[10+3*G.degree(n) for n in G.nodes()], color="#2a9d8f")))
    fig.update_layout(template=DARK, showlegend=False, height=560,
                      xaxis=dict(visible=False), yaxis=dict(visible=False),
                      title="Network Graph — Stakeholder Collaboration")
    fig.show()

# =============================================================================
# 12) SANKEY — value chain
# =============================================================================
labels = ["PGM ore","Refining","Catalysts (fuel cells)","Electrolysers",
          "Green H₂","Industrial use","Export","Beneficiation jobs"]
src = [0,1,1,2,3,3,4,5]
tgt = [1,2,3,4,5,6,7,7]
val = [100,70,30,60,40,20,35,55]
fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=20, color="#2a9d8f"),
    link=dict(source=src, target=tgt, value=val,
              color="rgba(42,157,143,0.35)")))
fig.update_layout(template=DARK, title="Sankey — PGM → Hydrogen Value Chain")
fig.show()

# =============================================================================
# 13) SUNBURST — projects by province
# =============================================================================
sb = pd.DataFrame({
    "Province":["Limpopo","Limpopo","Mpumalanga","Mpumalanga","Gauteng","Gauteng",
                "North West","North West"],
    "Project":["H₂ Valley","Fuel-cell pilot","Eskom microgrid","Catalyst plant",
               "R&D hub","Skills academy","Mine site A","Mine site B"],
    "Value":[120,60,180,75,140,55,90,40]})
fig = px.sunburst(sb, path=["Province","Project"], values="Value",
                  color="Value", color_continuous_scale="Viridis")
fig.update_layout(template=DARK, title="Sunburst — Hydrogen Projects by Province")
fig.show()

# =============================================================================
# 14) TREEMAP — investment allocation
# =============================================================================
tm = pd.DataFrame({
    "Category":["PGM mining","Refining","Catalyst mfg","Fuel cells",
                "Electrolysers","R&D","Skills","Export logistics"],
    "Investment_USDm":[420,180,260,310,290,90,45,150]})
fig = px.treemap(tm, path=["Category"], values="Investment_USDm",
                 color="Investment_USDm", color_continuous_scale="Viridis")
fig.update_layout(template=DARK, title="Treemap — Investment Allocation")
fig.show()

# =============================================================================
# 15) HEATMAP — industrial readiness
# =============================================================================
years_hm = list(range(2022, 2031))
segments = ["Mining","Refining","Catalyst","Fuel cell","H₂ production","Logistics"]
hm = np.cumsum(np.random.rand(len(segments), len(years_hm)) * 100, axis=1) \
     / np.arange(1, len(years_hm) + 1)
fig = go.Figure(go.Heatmap(z=hm, x=years_hm, y=segments,
                           colorscale="Inferno", colorbar=dict(title="Index")))
fig.update_layout(template=DARK, title="Heatmap — Industrial Readiness by Segment / Year")
fig.show()

# =============================================================================
# 16) SCATTER-MATRIX (SPLOM)
# =============================================================================
fig = px.scatter_matrix(
    df,
    dimensions=["PGM_Production_Moz","Hydrogen_Investment_USDbn",
                "PGM_Revenue_ZARbn","Jobs_Thousands","CO2_Reduction_MT"],
    color="Year")
fig.update_layout(template=DARK, height=760, title="Scatter Plot Matrix")
fig.show()

# =============================================================================
# 17) GEO MAP — project sites
# =============================================================================
sites = pd.DataFrame({
    "Site":["H₂ Valley (Limpopo)","Boegoebaai (NC)","Saldanha (WC)",
            "Coega (EC)","Richards Bay (KZN)","Johannesburg (GP)","Rustenburg (NW)"],
    "lat":[-23.9,-28.7,-33.0,-33.8,-28.8,-26.2,-25.7],
    "lon":[29.5,16.5,18.0,25.6,32.0,28.0,27.2],
    "Investment_USDm":[120,340,180,260,210,90,150]})
fig = px.scatter_geo(sites, lat="lat", lon="lon", size="Investment_USDm",
                     color="Site", text="Site", scope="africa",
                     projection="natural earth", size_max=45)
fig.update_traces(textposition="top center")
fig.update_layout(template=DARK, title="Scatter Map — SA Hydrogen Project Sites")
fig.show()

# =============================================================================
# 18) 3D SCATTER — investment × PGM × revenue
# =============================================================================
fig = px.scatter_3d(df, x="Hydrogen_Investment_USDbn", y="PGM_Production_Moz",
                    z="PGM_Revenue_ZARbn", color="Jobs_Thousands",
                    size="CO2_Reduction_MT", text="Year")
fig.update_layout(template=DARK,
                  title="3D Scatter — Investment × PGM × Revenue (size = CO₂)")
fig.show()

# =============================================================================
# 19) 3D YIELD CURVE SURFACE — H₂ yield vs T & P
# =============================================================================
temp = np.linspace(200, 800, 45)
press = np.linspace(1, 50, 45)
T, P = np.meshgrid(temp, press)
Z = 100 * np.exp(-((T - 500) / 220) ** 2) * (1 + 0.4 * np.sin(P / 10))
fig = go.Figure(go.Surface(x=temp, y=press, z=Z, colorscale="Viridis",
                           colorbar=dict(title="Yield %")))
fig.update_layout(template=DARK,
                  title="3D Yield-Curve Surface — H₂ yield vs T & P",
                  scene=dict(xaxis_title="T (K)",
                             yaxis_title="P (bar)",
                             zaxis_title="Yield (%)"))
fig.show()

# =============================================================================
# 20) WAFER THICKNESS CONTOUR — fuel-cell electrode substrate
# =============================================================================
xs = np.linspace(-1, 1, 80); ys = np.linspace(-1, 1, 80)
X, Y = np.meshgrid(xs, ys)
wafer = 200 + 25*np.exp(-3*(X**2 + Y**2)) + 8*np.sin(6*X)*np.cos(6*Y) \
        + np.random.randn(*X.shape) * 2
fig = go.Figure(go.Contour(x=xs, y=ys, z=wafer, colorscale="Jet",
                           contours=dict(showlabels=True)))
fig.update_layout(template=DARK,
                  title="Wafer Thickness Contour (µm) — fuel-cell electrode substrate")
fig.show()

# =============================================================================
# 21) COLLAR PLAN MAP — mine drill holes
# =============================================================================
holes = pd.DataFrame({
    "Hole":[f"DDH-{i:02d}" for i in range(1, 25)],
    "Easting":np.random.uniform(0, 1000, 24),
    "Northing":np.random.uniform(0, 1000, 24),
    "Depth_m":np.random.uniform(80, 420, 24),
    "PGM_grade_g_t":np.random.uniform(1.0, 6.5, 24)})
fig = px.scatter(holes, x="Easting", y="Northing", size="Depth_m",
                 color="PGM_grade_g_t", text="Hole",
                 color_continuous_scale="Plasma", size_max=30)
fig.update_traces(textposition="top center")
fig.update_layout(template=DARK,
                  title="Collar Plan Map — PGM Drilling (Bushveld Complex)",
                  yaxis=dict(scaleanchor="x", scaleratio=1))
fig.show()

# =============================================================================
# 22) AI / ML — KMeans clustering
# =============================================================================
if SKLEARN_OK:
    Xm = df[["Hydrogen_Investment_USDbn", "PGM_Production_Moz"]].values
    km = KMeans(n_clusters=3, n_init=10, random_state=42) \
         .fit(StandardScaler().fit_transform(Xm))
    df["Cluster"] = km.labels_
    fig = px.scatter(df, x="Hydrogen_Investment_USDbn", y="PGM_Production_Moz",
                     color="Cluster", text="Year",
                     color_continuous_scale="Turbo",
                     title="AI/ML — K-Means clusters of years (investment × PGM)")
    fig.update_traces(textposition="top center")
    fig.update_layout(template=DARK)
    fig.show()

# =============================================================================
# 23) PyTorch NN — jobs vs investment (optional)
# =============================================================================
if TORCH_OK:
    Xt = torch.tensor(df[["Hydrogen_Investment_USDbn"]].values, dtype=torch.float32)
    yt = torch.tensor(df[["Jobs_Thousands"]].values, dtype=torch.float32)
    net = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.05)
    losses = []
    for _ in range(500):
        opt.zero_grad()
        loss = ((net(Xt) - yt) ** 2).mean()
        loss.backward(); opt.step(); losses.append(loss.item())
    xx = torch.linspace(0, 16, 100).unsqueeze(1)
    yy = net(xx).detach().numpy().flatten()
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("Training loss", "NN fit"))
    fig.add_trace(go.Scatter(y=losses, mode="lines", name="loss"), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.Hydrogen_Investment_USDbn, y=df.Jobs_Thousands,
                             mode="markers", name="data"), row=1, col=2)
    fig.add_trace(go.Scatter(x=xx.numpy().flatten(), y=yy, mode="lines",
                             name="NN fit"), row=1, col=2)
    fig.update_layout(template=DARK, showlegend=False,
                      title="PyTorch — NN fit of Jobs vs Investment")
    fig.show()

# =============================================================================
# 24) BIOINFORMATICS — bioleaching gene expression heatmap
# =============================================================================
genes = ["cbbL","soxB","rus","cycA","hypA","hydA","nifH","cooA"]
conds = ["Day 1","Day 3","Day 7","Day 14","Day 21"]
expr = np.random.rand(len(genes), len(conds)) * 8
fig = go.Figure(go.Heatmap(z=expr, x=conds, y=genes, colorscale="YlGnBu",
                           colorbar=dict(title="log2 FC")))
fig.update_layout(template=DARK,
                  title="Bioinformatics — Bioleaching gene expression (PGM tailings)")
fig.show()

# =============================================================================
# 25) IMAGE DATA DISPLAY (imshow-style)
# =============================================================================
img = np.random.rand(64, 64)
fig = px.imshow(img, color_continuous_scale="Viridis")
fig.update_layout(template=DARK,
                  title="Image Data Display (imshow) — PGM SEM-style texture")
fig.show()

# =============================================================================
# 26) ANIMATION — investment × jobs × CO₂ over years
# =============================================================================
fig = px.scatter(df, x="Hydrogen_Investment_USDbn", y="Jobs_Thousands",
                 size="CO2_Reduction_MT", color="PGM_Revenue_ZARbn",
                 animation_frame="Year", text="Year",
                 range_x=[0, 17], range_y=[0, 110],
                 color_continuous_scale="Viridis")
fig.update_layout(template=DARK, height=520,
                  title="Animation — Investment × Jobs × CO₂ (2020–2030)")
fig.show()

# =============================================================================
# 27) SQL MOCK — filter + aggregate
# =============================================================================
sql_result = df[df.Year >= 2026][["Year", "PGM_Production_Moz",
                                  "Hydrogen_Investment_USDbn", "Jobs_Thousands"]]
print("\n=== SQL-style query — years ≥ 2026 ===")
print(sql_result.to_string(index=False))

# =============================================================================
# 28) JUPYTER WIDGETS + CUSTOM CONTROL
# =============================================================================
if IPY_OK:
    year_w = widgets.IntSlider(value=2026, min=2020, max=2030, step=1,
                               description="Year:")
    skill_w = widgets.Dropdown(options=list(skills_df.Skill),
                               value="Fuel Cell Tech.", description="Skill:")
    rate_w  = widgets.FloatSlider(value=0.10, min=0.05, max=0.20, step=0.01,
                                  description="Discount:")
    out = widgets.Output()

    def refresh(change=None):
        with out:
            out.clear_output()
            row = df[df.Year == year_w.value].iloc[0]
            gap = skills_df.loc[skills_df.Skill == skill_w.value, "Gap"].iloc[0]
            npv2 = mcp.call("roi_calc", capex=450,
                            cashflows=[60,85,110,130,145,150,155,160],
                            rate=rate_w.value)
            print(f"Year {year_w.value}: PGM {row.PGM_Production_Moz} Moz | "
                  f"H₂ ${row.Hydrogen_Investment_USDbn} bn | "
                  f"Jobs {row.Jobs_Thousands}k")
            print(f"Skills gap – {skill_w.value}: {int(gap)} specialists")
            print(f"Project NPV @ {rate_w.value:.0%}: USD {npv2:.1f} m")

    for w in (year_w, skill_w, rate_w):
        w.observe(refresh, names="value")
    safe_display(widgets.VBox([year_w, skill_w, rate_w, out]))
    refresh()

# =============================================================================
# 29) HMI / SCADA SNAPSHOT (mock)
# =============================================================================
hmi = pd.DataFrame({
    "Tag":["FT-101","PT-102","TT-103","AT-104","WT-105","EI-106"],
    "Description":["H₂ flow","Electrolyser pressure","Stack temperature",
                   "PGM catalyst purity","Cooling water","Power draw"],
    "Value":[128.4, 30.2, 68.1, 99.3, 45.0, 12.4],
    "Unit":["Nm³/h","bar","°C","%","m³/h","MW"],
    "Status":["OK","OK","WARN","OK","OK","OK"]})
rows = "".join(
    "<tr>" + "".join(
        f"<td style='border:1px solid #334155;padding:6px;"
        f"color:{'#f25c54' if r.Status=='WARN' else '#e6edf3'}'>{r[c]}</td>"
        for c in hmi.columns) + "</tr>"
    for _, r in hmi.iterrows())
hdr = "".join(f"<th style='border:1px solid #334155;padding:6px'>{c}</th>"
              for c in hmi.columns)
safe_html(f"<h3 style='color:#e6edf3'>SCADA / HMI — live snapshot (mock)</h3>"
          f"<table style='color:#e6edf3;border-collapse:collapse'>{hdr}{rows}</table>")

# =============================================================================
# 30) KUBERNETES (mock manifest for the MCP server)
# =============================================================================
k8s_yaml = textwrap.dedent("""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: pgm-hydrogen-mcp
spec:
  replicas: 3
  selector: {matchLabels: {app: pgm-mcp}}
  template:
    metadata: {labels: {app: pgm-mcp}}
    spec:
      containers:
      - name: mcp-server
        image: registry.mistra.ac.za/pgm-mcp:2026.1
        resources:
          limits: {cpu: "2", memory: "4Gi", nvidia.com/gpu: "1"}   # GPU accel
""")
print("\n=== Kubernetes manifest (mock) ===")
print(k8s_yaml)

# =============================================================================
# 31) MCP tool calls + LLM Q&A
# =============================================================================
print("=== Registered MCP tools ===")
for t in mcp.list_tools():
    print(" •", t["name"], "—", t["description"])
print("\nMCP:", mcp.call("get_pgm_production", year=2028))
print("MCP:", mcp.call("get_h2_investment", year=2030))
print("MCP:", mcp.call("skills_gap", skill="Fuel Cell Tech."))
print("MCP:", mcp.call("roi_calc", capex=450,
                      cashflows=[60,85,110,130,145,150,155,160], rate=0.10))

print("\n=== Mock LLM Q&A (grounded on MCP tools) ===")
for q in ["What's the investment outlook?",
          "Which skills gap is largest?",
          "Give an ROI estimate for the flagship project."]:
    print("Q:", q)
    print("A:", llm.respond(q), "\n")

print("Figma design tokens:", figma.get_design_tokens())
print("Figma frame layout:", figma.get_frame("PGM-Dashboard"))

# =============================================================================
# 32) amCharts (HTML embed)
# =============================================================================
am_html = """
<div style="background:#0e1116;padding:8px;border-radius:8px">
<h3 style="color:#ffb703;font-family:Inter">amCharts 5 — 2026–2030 H₂ investment (USD bn)</h3>
<div id="chartdiv" style="width:100%;height:280px"></div>
<script src="https://cdn.amcharts.com/lib/5/index.js"></script>
<script src="https://cdn.amcharts.com/lib/5/xy.js"></script>
<script>
 var root = am5.Root.new("chartdiv");
 root._logo.dispose();
 var chart = root.container.children.push(am5xy.XYChart.new(root, {}));
 var xAxis = chart.xAxes.push(am5xy.CategoryAxis.new(root,
              {categoryField:"year", renderer: am5xy.AxisRendererX.new(root,{})}));
 var yAxis = chart.yAxes.push(am5xy.ValueAxis.new(root,
              {renderer: am5xy.AxisRendererY.new(root,{})}));
 var series = chart.series.push(am5xy.ColumnSeries.new(root,
              {xAxis:xAxis, yAxis:yAxis, valueYField:"value",
               categoryXField:"year", sequencedInterpolation: true}));
 series.columns.template.setAll({fill: am5.color(0x2a9d8f)});
 var data = [{year:"2026",value:5.0},{year:"2027",value:7.2},
             {year:"2028",value:9.8},{year:"2029",value:12.5},
             {year:"2030",value:15.0}];
 xAxis.data.setAll(data); series.data.setAll(data);
 series.appear(800); chart.appear(800,100);
</script>
<!-- SofaAir-style embed / Google-Studio-style KPI cards are proxied via the HTML
     header above; Dash Enterprise / Plotly Studio would host these charts. -->
"""
safe_html(am_html)

# =============================================================================
# END
# =============================================================================
print("\n✔ Demo complete — all layers exercised:")
print("  MCP → LLM → Figma-MCP → Plotly Studio → Dash Enterprise → amCharts → HTML/K8s/HMI")


=== SQL-style query — years ≥ 2026 ===
 Year  PGM_Production_Moz  Hydrogen_Investment_USDbn  Jobs_Thousands
 2026                 4.4                        5.0              44
 2027                 4.6                        7.2              55
 2028                 4.8                        9.8              68
 2029                 5.0                       12.5              82
 2030                 5.2                       15.0              98


FT-101,H₂ flow,128.4,Nm³/h,OK
PT-102,Electrolyser pressure,30.2,bar,OK
TT-103,Stack temperature,68.1,°C,WARN
AT-104,PGM catalyst purity,99.3,%,OK
WT-105,Cooling water,45.0,m³/h,OK
EI-106,Power draw,12.4,MW,OK



=== Kubernetes manifest (mock) ===

apiVersion: apps/v1
kind: Deployment
metadata:
  name: pgm-hydrogen-mcp
spec:
  replicas: 3
  selector: {matchLabels: {app: pgm-mcp}}
  template:
    metadata: {labels: {app: pgm-mcp}}
    spec:
      containers:
      - name: mcp-server
        image: registry.mistra.ac.za/pgm-mcp:2026.1
        resources:
          limits: {cpu: "2", memory: "4Gi", nvidia.com/gpu: "1"}   # GPU accel

=== Registered MCP tools ===
 • get_pgm_production — PGM production (Moz) for a given year
 • get_h2_investment — Hydrogen investment (USD bn) for a given year
 • skills_gap — 2030 skills gap for a named occupation
 • roi_calc — NPV-style ROI over cashflows

MCP: 4.8
MCP: 15.0
MCP: 580.0
MCP: 175.1151362373455

=== Mock LLM Q&A (grounded on MCP tools) ===
Q: What's the investment outlook?
A: Total projected H₂ investment 2020–2030 ≈ USD 57.2 bn. Biggest single-year jump is 2029→2030 ($12.5 → $15.0 bn). 

Q: Which skills gap is largest?
A: Largest skills gap is **Chemi


✔ Demo complete — all layers exercised:
  MCP → LLM → Figma-MCP → Plotly Studio → Dash Enterprise → amCharts → HTML/K8s/HMI


In [6]:

# ============================================================================
#  MISTRA 2026 ANNUAL PGM ROUNDTABLE
#  "From Strategic Aspirations to Execution Reality:
#   Accelerating South Africa's PGM-Powered Hydrogen Economy"
#  ---------------------------------------------------------------------------
#  ONE-CELL PYTHON VISUAL ANALYTICS DASHBOARD (synthetic / demo data)
#  Requires:  pip install numpy pandas plotly networkx
#  Covers: data table, basic + statistical charts, scientific charts,
#          time-series subplots, sankey, sunburst, treemap, heatmap,
#          radar, scatter-matrix, scatter-map (geo), 3-D scatter,
#          3-D yield-curve surface, contour (wafer/catalyst thickness),
#          network graph (MCP-style stakeholder graph), financial
#          waterfall + candlestick, animation with custom controls,
#          exportable single HTML (open-source, no token needed).
# ============================================================================

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

RNG = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# 1. SYNTHETIC DATA LAYER  (replace with SQL / MCP server / live feed)
# ---------------------------------------------------------------------------
prov_coords = {
    'Limpopo': (-23.9, 29.4), 'North West': (-25.8, 25.6), 'Gauteng': (-26.2, 28.0),
    'Mpumalanga': (-25.5, 30.5), 'Free State': (-28.5, 26.5), 'KwaZulu-Natal': (-29.0, 30.5),
    'Eastern Cape': (-32.0, 26.0), 'Western Cape': (-33.5, 22.0), 'Northern Cape': (-29.0, 21.0)
}

projects = pd.DataFrame({
    'Project': [
        'Platinum Valley H2 Hub', 'Bushveld Electrolyser Cluster', 'Coega Green H2',
        'Boegoebaai H2 Gateway', 'Saldanha Fuel-Cell Plant', 'Richards Bay Green Ammonia',
        'Secunda CCUS-H2 Retrofit', 'Rustenburg PGM Catalysts', 'Limpopo Fuel-Cell Microgrid',
        'Mpumalanga H2 Truck Corridor', 'Gauteng H2 Skills Academy', 'North West Refinery Upgrade',
        'Free State H2-Agri Cluster', 'E Cape Fuel-Cell Backup', 'W Cape Marine H2 Bunkering',
        'N Cape Solar-H2 Gigaplant', 'KZN Port H2 Bunkering', 'Gauteng PGM Beneficiation Park'],
    'Province': ['Limpopo', 'North West', 'Eastern Cape', 'Northern Cape', 'Western Cape',
                 'KwaZulu-Natal', 'Mpumalanga', 'North West', 'Limpopo', 'Mpumalanga', 'Gauteng',
                 'North West', 'Free State', 'Eastern Cape', 'Western Cape', 'Northern Cape',
                 'KwaZulu-Natal', 'Gauteng'],
    'Sector': ['Mining & Refining', 'Electrolysis', 'Ammonia/Export', 'Ammonia/Export', 'Fuel Cells',
               'Ammonia/Export', 'Infrastructure', 'Beneficiation', 'Fuel Cells', 'Mobility',
               'Skills/Human Capital', 'Mining & Refining', 'Mobility', 'Fuel Cells', 'Infrastructure',
               'Electrolysis', 'Infrastructure', 'Beneficiation'],
    'Status': ['Operational', 'Construction', 'FID', 'Feasibility', 'Construction', 'FID',
               'Operational', 'Operational', 'Construction', 'Feasibility', 'Operational', 'FID',
               'Feasibility', 'Concept', 'FID', 'Feasibility', 'Concept', 'Construction'],
    'Capex_ZARbn': [12.5, 8.2, 45.0, 38.6, 6.4, 52.3, 9.8, 3.2, 1.6, 4.5, 0.9, 7.4, 2.8, 1.2, 5.6, 62.0, 3.4, 4.1],
    'Capacity_MW': [0, 320, 1200, 1000, 150, 1500, 250, 0, 25, 80, 0, 0, 40, 15, 100, 2000, 60, 0],
    'PGM_koz_yr': [1200, 85, 0, 0, 120, 0, 40, 260, 18, 22, 0, 300, 8, 6, 30, 0, 12, 190],
    'IRR_pct': [14.2, 11.8, 16.5, 17.2, 12.4, 18.1, 9.6, 21.4, 10.2, 13.5, 8.0, 15.1, 11.0, 9.4, 12.8, 19.3, 10.6, 22.0],
    'Jobs': [4200, 1800, 6500, 5200, 900, 7400, 1100, 600, 350, 800, 250, 1500, 500, 220, 950, 8800, 600, 1200],
    'COD_Year': [2024, 2026, 2029, 2030, 2027, 2030, 2025, 2024, 2026, 2028, 2024, 2027, 2028, 2027, 2028, 2031, 2029, 2026],
})
projects['Lat'] = projects['Province'].map(lambda p: prov_coords[p][0]) + RNG.normal(0, 0.55, len(projects))
projects['Lon'] = projects['Province'].map(lambda p: prov_coords[p][1]) + RNG.normal(0, 0.55, len(projects))
projects['COD_str'] = projects['COD_Year'].astype(str)

years = np.arange(2020, 2036)
ts = pd.DataFrame({
    'Year': years,
    'Investment_ZARbn': [0.8, 1.1, 1.5, 2.0, 2.6, 3.4, 4.5, 6.0, 8.0, 10.5, 13.0, 15.5, 17.0, 18.0, 18.5, 19.0],
    'PGM_Production_Moz': [4.9, 5.0, 4.6, 4.4, 4.5, 4.7, 4.8, 5.0, 5.1, 5.2, 5.3, 5.4, 5.5, 5.5, 5.6, 5.6],
    'H2_Capacity_MW': [20, 45, 90, 160, 280, 450, 700, 1050, 1500, 2100, 2900, 3800, 4700, 5600, 6400, 7100],
    'Jobs_Supported': [1200, 1800, 2600, 3600, 5000, 6800, 9200, 12500, 16000, 20500, 25000, 29500, 33500, 36500, 39000, 41000],
})

skills = ['Electrochemistry', 'Control & Instrumentation', 'Metallurgy', 'Hydrogen Safety',
          'Project Finance', 'Data / ML & AI', 'Maintenance & Ops', 'Systems Integration']
skill_matrix = pd.DataFrame(
    RNG.integers(20, 100, size=(len(prov_coords), len(skills))),
    index=list(prov_coords.keys()), columns=skills)

pillars = ['Policy & Regulation', 'Investment & Finance', 'Skills & Workforce', 'Infrastructure',
           'Technology & R&D', 'Beneficiation', 'Collaboration', 'Market Access']
radar_2024 = [72, 48, 41, 38, 62, 45, 52, 55]
radar_2026 = [85, 68, 66, 61, 78, 70, 80, 74]

# ---------------------------------------------------------------------------
# 2. FIGURE BUILDERS
# ---------------------------------------------------------------------------
FIGS = []
def add(title, fig):
    FIGS.append((title, fig))

# ---- 2.1 Synthetic project register (DATA TABLE) --------------------------
fig_table = go.Figure(data=[go.Table(
    columnwidth=[150, 95, 115, 85, 80, 80, 75, 65, 65],
    header=dict(values=['Project', 'Province', 'Sector', 'Status', 'Capex (ZAR bn)',
                        'Capacity (MW)', 'PGM (koz/yr)', 'IRR %', 'COD'],
                fill_color='#0b3d6b', font=dict(color='white', size=12), align='left', height=30),
    cells=dict(values=[projects['Project'], projects['Province'], projects['Sector'],
                       projects['Status'], projects['Capex_ZARbn'].round(1),
                       projects['Capacity_MW'], projects['PGM_koz_yr'],
                       projects['IRR_pct'].round(1), projects['COD_Year']],
               fill_color=[['#f3f7fb' if i % 2 == 0 else '#e6eef7' for i in range(len(projects))]],
               align='left', font=dict(size=11), height=24))])
fig_table.update_layout(title='Synthetic PGM–Hydrogen project register (18 pipeline assets)',
                        margin=dict(l=10, r=10, t=50, b=10), height=620)
add('1 · Synthetic project register (data table)', fig_table)

# ---- 2.2 TIME-SERIES SUBPLOTS --------------------------------------------
fig_ts = make_subplots(rows=2, cols=2, shared_xaxes=False,
                       subplot_titles=('Hydrogen & PGM investment (ZAR bn)',
                                       'PGM production (Moz)',
                                       'Installed H2 capacity (MW)',
                                       'Jobs supported'))
specs = [('Investment_ZARbn', '#0b3d6b', 1, 1), ('PGM_Production_Moz', '#b8860b', 1, 2),
         ('H2_Capacity_MW', '#2e8b57', 2, 1), ('Jobs_Supported', '#8b008b', 2, 2)]
for col, colour, r, c in specs:
    fig_ts.add_trace(go.Scatter(x=ts['Year'], y=ts[col], mode='lines+markers',
                                line=dict(color=colour, width=3), marker=dict(size=6),
                                name=col, showlegend=False,
                                fill='tozeroy', fillcolor=colour.replace('#', 'rgba(') if False else None),
                     row=r, col=c)
fig_ts.update_layout(title='Macro trajectory 2020–2035 (time-series subplots)',
                     template='plotly_white', height=620, margin=dict(t=70))
add('2 · Time-series subplots', fig_ts)

# ---- 2.3 SANKEY — PGM → H2 value chain -----------------------------------
sankey_nodes = ['PGM Ore', 'PGM Concentrate', 'Refined PGM', 'Catalysts & MEAs', 'Electrolysers',
                'Fuel Cells', 'Green H2', 'Green Ammonia', 'H2 Mobility', 'Industrial Heat',
                'Power & Backup', 'Exports', 'Domestic Beneficiation']
sankey_links = [(0, 1, 100), (1, 2, 96), (2, 3, 34), (2, 11, 26), (2, 12, 36), (3, 4, 19),
                (3, 5, 15), (4, 6, 18), (5, 10, 13), (5, 8, 2), (6, 7, 10), (6, 8, 4),
                (6, 9, 4), (7, 11, 8), (7, 9, 2), (9, 12, 3), (10, 12, 4), (8, 12, 3)]
fig_sankey = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(label=sankey_nodes, pad=18, thickness=18,
              color=['#8b6914', '#a67c00', '#d4af37', '#2e8b57', '#3cb371', '#66cdaa',
                     '#1f77b4', '#17becf', '#0b3d6b', '#7f8c8d', '#95a5a6', '#c0392b', '#e67e22']),
    link=dict(source=[l[0] for l in sankey_links], target=[l[1] for l in sankey_links],
              value=[l[2] for l in sankey_links],
              color='rgba(120,150,190,0.35)')))
fig_sankey.update_layout(title='PGM → hydrogen value-chain flows (Sankey, indexed units)',
                         font=dict(size=11), height=620, margin=dict(t=60))
add('3 · Sankey — PGM to hydrogen value chain', fig_sankey)

# ---- 2.4 SUNBURST — capital allocation hierarchy -------------------------
sb_labels = ['H2 & PGM Economy',
             'Public Capital', 'Private Capital', 'DFI / Concessional', 'Grants & PPP',
             'Infrastructure', 'Skills', 'R&D', 'Regulatory',
             'Mining / Refining', 'Electrolysers', 'Fuel Cells', 'Mobility',
             'IDC / DBSA', 'IFIs', 'Green Bonds',
             'EU / GIZ', 'PPP', 'Academia']
sb_parents = ['', 'H2 & PGM Economy', 'H2 & PGM Economy', 'H2 & PGM Economy', 'H2 & PGM Economy',
              'Public Capital', 'Public Capital', 'Public Capital', 'Public Capital',
              'Private Capital', 'Private Capital', 'Private Capital', 'Private Capital',
              'DFI / Concessional', 'DFI / Concessional', 'DFI / Concessional',
              'Grants & PPP', 'Grants & PPP', 'Grants & PPP']
sb_values = [160, 42, 68, 35, 15,
             18, 9, 8, 7, 28, 16, 12, 12, 18, 12, 5, 6, 5, 4]
fig_sun = go.Figure(go.Sunburst(labels=sb_labels, parents=sb_parents, values=sb_values,
                                branchvalues='total', maxdepth=2,
                                marker=dict(colorscale='Blues')))
fig_sun.update_layout(title='Capital allocation hierarchy (Sunburst, ZAR bn)', height=620, margin=dict(t=60))
add('4 · Sunburst — capital allocation', fig_sun)

# ---- 2.5 TREEMAP — capex by province / sector ----------------------------
fig_tree = px.treemap(projects, path=[px.Constant('South Africa'), 'Province', 'Sector'],
                      values='Capex_ZARbn', color='Capex_ZARbn',
                      color_continuous_scale='Turbo', hover_data=['Project', 'IRR_pct'])
fig_tree.update_layout(title='Project capex treemap — province → sector (ZAR bn)', height=620, margin=dict(t=60))
add('5 · Treemap — capex by province and sector', fig_tree)

# ---- 2.6 HEATMAP — skills demand by province -----------------------------
fig_heat = go.Figure(go.Heatmap(z=skill_matrix.values, x=skill_matrix.columns,
                                y=skill_matrix.index, colorscale='YlOrRd',
                                text=skill_matrix.values, texttemplate='%{text}',
                                colorbar=dict(title='Demand index')))
fig_heat.update_layout(title='Skills demand heatmap — province × capability (statistical chart)',
                       height=620, margin=dict(t=60, b=120))
add('6 · Heatmap — skills demand', fig_heat)

# ---- 2.7 SCATTER-PLOT MATRIX --------------------------------------------
fig_sm = px.scatter_matrix(projects, dimensions=['Capex_ZARbn', 'Capacity_MW', 'IRR_pct', 'Jobs'],
                           color='Sector', hover_name='Project', height=680)
fig_sm.update_traces(diagonal_visible=False, showupperhalf=False, marker=dict(size=7))
fig_sm.update_layout(title='Scatter-plot matrix — project economics', margin=dict(t=70))
add('7 · Scatter-plot matrix', fig_sm)

# ---- 2.8 3-D SCATTER ------------------------------------------------------
fig_3d = px.scatter_3d(projects, x='Capex_ZARbn', y='Capacity_MW', z='IRR_pct',
                       size='Jobs', color='Province', symbol='Status',
                       hover_name='Project', size_max=45, opacity=0.85,
                       color_discrete_sequence=px.colors.qualitative.Bold)
fig_3d.update_layout(title='3-D project scatter — capex × capacity × IRR (bubble = jobs)',
                     scene=dict(xaxis_title='Capex (ZAR bn)', yaxis_title='Capacity (MW)',
                                zaxis_title='IRR (%)'), height=680)
add('8 · 3-D scatter', fig_3d)

# ---- 2.9 3-D YIELD-CURVE SURFACE -----------------------------------------
px_ = np.linspace(60, 160, 60)
py_ = np.linspace(-10, 40, 60)
X, Y = np.meshgrid(px_, py_)
Z = 6 + 0.11 * (X - 60) - 0.15 * Y + 0.0008 * (X - 60) * Y
fig_surf = go.Figure(go.Surface(x=px_, y=py_, z=Z, colorscale='Viridis',
                                colorbar=dict(title='IRR %')))
fig_surf.update_layout(title='3-D yield (IRR) surface — PGM basket price vs capex overrun',
                       scene=dict(xaxis_title='PGM basket price index',
                                  yaxis_title='Capex overrun (%)',
                                  zaxis_title='Project IRR (%)'), height=680)
add('9 · 3-D yield-curve surface', fig_surf)

# ---- 2.10 CONTOUR — catalyst “wafer” thickness vs temperature ------------
tx = np.linspace(20, 200, 120)
ty = np.linspace(60, 120, 120)
TX, TY = np.meshgrid(tx, ty)
TZ = 45 + 18 * np.exp(-(((TX - 100) ** 2) / (2 * 40 ** 2) + ((TY - 85) ** 2) / (2 * 18 ** 2)))
fig_cont = go.Figure()
fig_cont.add_trace(go.Contour(x=tx, y=ty, z=TZ, colorscale='Turbo',
                              contours=dict(coloring='heatmap', showlabels=True,
                                            labelfont=dict(size=9, color='white'))))
fig_cont.add_trace(go.Scatter(x=[100], y=[85], mode='markers+text', text=['Design point'],
                              textposition='top center',
                              marker=dict(size=14, color='white', line=dict(color='black', width=2)),
                              showlegend=False))
fig_cont.update_layout(title='Contour map — catalyst-layer thickness × operating temperature → efficiency (%)',
                       xaxis_title='Catalyst / membrane thickness (µm)',
                       yaxis_title='Operating temperature (°C)', height=640, margin=dict(t=70))
add('10 · Contour — thickness vs temperature', fig_cont)

# ---- 2.11 RADAR — readiness pillars --------------------------------------
fig_radar = go.Figure()
fig_radar.add_trace(go.Scatterpolar(r=radar_2024 + [radar_2024[0]],
                                    theta=pillars + [pillars[0]], fill='toself',
                                    name='2024 baseline', line=dict(color='#c0392b')))
fig_radar.add_trace(go.Scatterpolar(r=radar_2026 + [radar_2026[0]],
                                    theta=pillars + [pillars[0]], fill='toself',
                                    name='2026 target', line=dict(color='#0b3d6b')))
fig_radar.update_layout(title='Execution-readiness radar — strategy → implementation',
                        polar=dict(radialaxis=dict(visible=True, range=[0, 100])), height=640)
add('11 · Radar — execution readiness', fig_radar)

# ---- 2.12 NETWORK GRAPH — stakeholder collaboration (MCP-style) ----------
net_nodes = [('DSTI', 'Government'), ('DMRE', 'Government'), ('DTIC', 'Government'),
             ('MISTRA', 'Think Tank'), ('Anglo American', 'Mining'), ('Sibanye-Stillwater', 'Mining'),
             ('Implats', 'Mining'), ('Valterra', 'Industry'), ('HyPlat', 'Industry'),
             ('Bambili', 'Industry'), ('Sasol', 'Industry'), ('Air Products', 'Industry'),
             ('NECSA', 'SOE'), ('CSIR', 'Academia'), ('Mintek', 'Academia'), ('TUT', 'Academia'),
             ('Wits', 'Academia'), ('UJ', 'Academia'), ('IDC', 'DFI'), ('DBSA', 'DFI'),
             ('Nedbank', 'Finance'), ('Standard Bank', 'Finance'), ('Eskom', 'SOE'),
             ('Transnet', 'SOE'), ('Coega SEZ', 'Infrastructure'), ('Richards Bay IDZ', 'Infrastructure')]
net_edges = [('DSTI', 'MISTRA', 3), ('DSTI', 'CSIR', 3), ('DSTI', 'NECSA', 2), ('DSTI', 'HyPlat', 2),
             ('DMRE', 'Anglo American', 3), ('DMRE', 'Sibanye-Stillwater', 3), ('DMRE', 'Implats', 2),
             ('DTIC', 'Coega SEZ', 2), ('DTIC', 'Richards Bay IDZ', 2), ('DTIC', 'Bambili', 2),
             ('MISTRA', 'Anglo American', 2), ('MISTRA', 'IDC', 2), ('MISTRA', 'Wits', 2), ('MISTRA', 'TUT', 2),
             ('Anglo American', 'Valterra', 3), ('Anglo American', 'HyPlat', 2), ('Sibanye-Stillwater', 'HyPlat', 3),
             ('Implats', 'HyPlat', 2), ('Valterra', 'Bambili', 3), ('HyPlat', 'Bambili', 2),
             ('Sasol', 'Air Products', 3), ('Sasol', 'Transnet', 2), ('Air Products', 'Coega SEZ', 2),
             ('CSIR', 'Mintek', 3), ('Mintek', 'HyPlat', 2), ('NECSA', 'CSIR', 2),
             ('TUT', 'UJ', 2), ('Wits', 'TUT', 2), ('Wits', 'CSIR', 2), ('UJ', 'Bambili', 2),
             ('IDC', 'DBSA', 2), ('IDC', 'Valterra', 2), ('DBSA', 'Coega SEZ', 2), ('DBSA', 'Richards Bay IDZ', 2),
             ('Nedbank', 'Anglo American', 2), ('Standard Bank', 'Sasol', 2), ('Nedbank', 'IDC', 2),
             ('Eskom', 'Sasol', 2), ('Eskom', 'Coega SEZ', 2), ('Transnet', 'Richards Bay IDZ', 3),
             ('Coega SEZ', 'Bambili', 2), ('Richards Bay IDZ', 'Air Products', 2)]
G = nx.Graph()
G.add_nodes_from([n for n, _ in net_nodes])
G.add_weighted_edges_from(net_edges)
pos = nx.spring_layout(G, seed=7, k=1.15, iterations=300)
cat_of = dict(net_nodes)
cat_colours = {'Government': '#1f4e79', 'Mining': '#b8860b', 'Industry': '#2e8b57',
               'Think Tank': '#8b008b', 'SOE': '#c0392b', 'Academia': '#2980b9',
               'DFI': '#e67e22', 'Finance': '#16a085', 'Infrastructure': '#7f8c8d'}
ex, ey = [], []
for u, v, w in net_edges:
    ex += [pos[u][0], pos[v][0], None]
    ey += [pos[u][1], pos[v][1], None]
fig_net = go.Figure()
fig_net.add_trace(go.Scatter(x=ex, y=ey, mode='lines', hoverinfo='none',
                             line=dict(color='#c8d3de', width=1.6), showlegend=False))
for cat, colour in cat_colours.items():
    xs = [pos[n][0] for n in G.nodes if cat_of[n] == cat]
    ys = [pos[n][1] for n in G.nodes if cat_of[n] == cat]
    ns = [n for n in G.nodes if cat_of[n] == cat]
    degs = [G.degree(n) for n in ns]
    fig_net.add_trace(go.Scatter(x=xs, y=ys, mode='markers+text', text=ns, name=cat,
                                 textposition='top center', textfont=dict(size=9),
                                 marker=dict(size=[16 + 5 * d for d in degs], color=colour,
                                             line=dict(color='white', width=1.5)),
                                 hovertext=[f'{n} (degree {d})' for n, d in zip(ns, degs)]))
fig_net.update_layout(title='Stakeholder collaboration network (MCP-style graph) — government · industry · academia · finance',
                      xaxis=dict(visible=False), yaxis=dict(visible=False),
                      template='plotly_white', height=700, margin=dict(t=70))
add('12 · Network graph — collaboration', fig_net)

# ---- 2.13 GEO MAP — project bubble map ----------------------------------
fig_map = go.Figure(go.Scattergeo(
    lon=projects['Lon'], lat=projects['Lat'], text=projects['Project'],
    customdata=np.stack([projects['Province'], projects['Capex_ZARbn'], projects['IRR_pct']], axis=-1),
    hovertemplate='<b>%{text}</b><br>%{customdata[0]}<br>Capex: ZAR %{customdata[1]:.1f} bn<br>IRR: %{customdata[2]:.1f}%<extra></extra>',
    marker=dict(size=6 + projects['Capex_ZARbn'] * 0.9,
                color=projects['IRR_pct'], colorscale='Turbo', showscale=True,
                colorbar=dict(title='IRR %'), line=dict(color='white', width=1), opacity=0.85)))
fig_map.update_geos(scope='africa', showcountries=True, countrycolor='lightgrey',
                    showland=True, landcolor='#f2f5f8', showocean=True, oceancolor='#e6f0f7',
                    lataxis_range=[-36, -20], lonaxis_range=[14, 34])
fig_map.update_layout(title='Project geography — bubble map of PGM–hydrogen assets',
                      height=700, margin=dict(t=60, b=0, l=0, r=0))
add('13 · Scatter map — project geography', fig_map)

# ---- 2.14 FINANCIAL CHARTS — waterfall + candlestick --------------------
months = pd.date_range('2023-01-01', periods=36, freq='MS')
base = 100 + np.cumsum(RNG.normal(0, 2.4, 36))
opn = base + RNG.normal(0, 1.2, 36)
cls = base + RNG.normal(0, 1.2, 36)
hi = np.maximum(opn, cls) + np.abs(RNG.normal(1.5, 0.5, 36))
lo = np.minimum(opn, cls) - np.abs(RNG.normal(1.5, 0.5, 36))
fig_fin = make_subplots(rows=1, cols=2, subplot_titles=(
    'Typical project funding stack (waterfall, ZAR bn)',
    'PGM basket price index (candlestick, 36 months)'),
    specs=[[{'type': 'waterfall'}, {'type': 'xy'}]], column_widths=[0.45, 0.55])
fig_fin.add_trace(go.Waterfall(
    x=['Total Capex', 'Equity', 'DFI Debt', 'Commercial Debt', 'Green Bonds', 'Grants', 'Unfunded Gap'],
    measure=['absolute', 'relative', 'relative', 'relative', 'relative', 'relative', 'total'],
    y=[10.0, -3.0, -2.5, -1.8, -1.2, -0.5, 0],
    text=['10.0', '-3.0', '-2.5', '-1.8', '-1.2', '-0.5', '1.0'], textposition='outside',
    connector=dict(line=dict(color='#999')), increasing=dict(marker_color='#2e8b57'),
    decreasing=dict(marker_color='#c0392b'), totals=dict(marker_color='#0b3d6b')),
    row=1, col=1)
fig_fin.add_trace(go.Candlestick(x=months, open=opn, high=hi, low=lo, close=cls,
                                 increasing_line_color='#2e8b57',
                                 decreasing_line_color='#c0392b', name='PGM basket'),
                  row=1, col=2)
fig_fin.update_layout(title='Financial charts — funding waterfall & PGM price candlestick',
                      showlegend=False, height=620, xaxis2_rangeslider_visible=False)
add('14 · Financial charts — waterfall & candlestick', fig_fin)

# ---- 2.15 ANIMATION with custom controls --------------------------------
fig_anim = px.scatter(projects, x='Capex_ZARbn', y='Jobs', size='Capacity_MW',
                      size_max=60, color='Sector', animation_frame='COD_str',
                      hover_name='Project', range_x=[0, 70], range_y=[0, 10000],
                      color_discrete_sequence=px.colors.qualitative.Set2)
fig_anim.update_layout(title='Animated project pipeline by commercial operation date (custom slider control)',
                       template='plotly_white', height=640,
                       xaxis_title='Capex (ZAR bn)', yaxis_title='Jobs')
add('15 · Animation — pipeline over time', fig_anim)

# ---- 2.16 MASTER DASHBOARD with dropdown custom control ----------------
prov_capex = projects.groupby('Province')['Capex_ZARbn'].sum().sort_values(ascending=False)
prov_jobs = projects.groupby('Province')['Jobs'].sum().reindex(prov_capex.index)
prov_cap = projects.groupby('Province')['Capacity_MW'].sum().reindex(prov_capex.index)
sector_capex = projects.groupby('Sector')['Capex_ZARbn'].sum()
status_counts = projects['Status'].value_counts()

fig_master = make_subplots(rows=2, cols=2,
                           specs=[[{'type': 'xy'}, {'type': 'domain'}],
                                  [{'type': 'xy'}, {'type': 'xy'}]],
                           subplot_titles=('Provincial comparison (use dropdown)',
                                           'Capex share by sector',
                                           'Cumulative capex & H2 capacity',
                                           'Pipeline by status'))
fig_master.add_trace(go.Bar(x=prov_capex.index, y=prov_capex.values, name='Capex (ZAR bn)',
                            marker_color='#0b3d6b', visible=True), row=1, col=1)
fig_master.add_trace(go.Bar(x=prov_jobs.index, y=prov_jobs.values, name='Jobs',
                            marker_color='#b8860b', visible=False), row=1, col=1)
fig_master.add_trace(go.Bar(x=prov_cap.index, y=prov_cap.values, name='Capacity (MW)',
                            marker_color='#2e8b57', visible=False), row=1, col=1)
fig_master.add_trace(go.Pie(labels=sector_capex.index, values=sector_capex.values, hole=0.42,
                            textinfo='percent+label', textfont=dict(size=10), showlegend=False),
                     row=1, col=2)
fig_master.add_trace(go.Scatter(x=ts['Year'], y=np.cumsum(ts['Investment_ZARbn']), mode='lines',
                                name='Cumulative capex (ZAR bn)', line=dict(color='#0b3d6b', width=3)),
                     row=2, col=1)
fig_master.add_trace(go.Scatter(x=ts['Year'], y=ts['H2_Capacity_MW'], mode='lines+markers',
                                name='H2 capacity (MW)', line=dict(color='#2e8b57', width=2, dash='dot'),
                                yaxis='y4'), row=2, col=1)
fig_master.add_trace(go.Bar(x=status_counts.index, y=status_counts.values,
                            marker_color='#8b008b', showlegend=False), row=2, col=2)

fig_master.update_layout(
    title='Execution dashboard — custom dropdown control + dual axis',
    height=760, template='plotly_white',
    updatemenus=[dict(type='dropdown', direction='down', x=1.0, y=1.18, xanchor='right',
                      showactive=True, buttons=[
                          dict(label='Capex (ZAR bn)', method='update',
                               args=[{'visible': [True, False, False, True, True, True, True, True]}]),
                          dict(label='Jobs', method='update',
                               args=[{'visible': [False, True, False, True, True, True, True, True]}]),
                          dict(label='Capacity (MW)', method='update',
                               args=[{'visible': [False, False, True, True, True, True, True, True]}])])],
    yaxis4=dict(title='MW', overlaying='y3', side='right', showgrid=False))
fig_master.update_yaxes(title_text='ZAR bn / units', row=1, col=1)
fig_master.update_yaxes(title_text='Cumulative ZAR bn', row=2, col=1)
add('16 · Master dashboard — dropdown controls', fig_master)

# ---------------------------------------------------------------------------
# 3. KPI SUMMARY + EXPORT TO A SINGLE STANDALONE HTML DASHBOARD
# ---------------------------------------------------------------------------
kpi = {
    'Pipeline capex (ZAR bn)': round(projects['Capex_ZARbn'].sum(), 1),
    'Pipeline H2 capacity (MW)': int(projects['Capacity_MW'].sum()),
    'PGM feedstock (koz/yr)': int(projects['PGM_koz_yr'].sum()),
    'Jobs supported': int(projects['Jobs'].sum()),
    'Weighted average IRR (%)': round(np.average(projects['IRR_pct'], weights=projects['Capex_ZARbn']), 1),
    'Assets in register': len(projects),
}
print('=' * 78)
print('MISTRA 2026 ANNUAL PGM ROUNDTABLE — EXECUTION DASHBOARD KPIs')
print('=' * 78)
for k, v in kpi.items():
    print(f'{k:<32}: {v:>10}')

kpi_html = ''.join(
    f"<div class='kpi'><div style='font-size:12px;color:#5a6b7b'>{k}</div>"
    f"<div style='font-size:22px;font-weight:700;color:#0b3d6b'>{v}</div></div>"
    for k, v in kpi.items())

parts = [
    "<html><head><meta charset='utf-8'><title>MISTRA 2026 PGM Roundtable Dashboard</title>",
    "<style>body{font-family:Segoe UI,Arial,sans-serif;margin:20px;background:#fbfbfd;color:#12263a}",
    "h1{color:#0b3d6b;font-size:26px}h2{color:#12507f;border-bottom:2px solid #d6e4f0;",
    "padding-bottom:5px;margin-top:40px;font-size:19px}",
    ".kpi{display:inline-block;background:#fff;border-radius:10px;padding:12px 20px;margin:6px;",
    "box-shadow:0 1px 5px rgba(0,0,0,.12);min-width:170px}",
    ".sub{color:#5a6b7b;font-size:14px;margin-bottom:14px}</style></head><body>",
    "<h1>MISTRA 2026 Annual PGM Roundtable &mdash; PGM-Powered Hydrogen Economy Dashboard</h1>",
    "<div class='sub'>From strategic aspirations to execution reality &middot; synthetic demonstration data</div>",
    kpi_html,
]
for i, (title, f) in enumerate(FIGS):
    parts.append(f"<h2>{title}</h2>")
    parts.append(f.to_html(full_html=False, include_plotlyjs=('cdn' if i == 0 else False),
                           config={'responsive': True, 'displaylogo': False}))
parts.append("</body></html>")
html_doc = ''.join(parts)

out = 'mistra_2026_pgm_hydrogen_dashboard.html'
with open(out, 'w', encoding='utf-8') as fh:
    fh.write(html_doc)
print(f'\nSaved standalone dashboard -> {out}  ({len(html_doc)/1e6:.2f} MB)')

# Inline render (Jupyter / Colab). Comment out to avoid a very large cell output.
try:
    from IPython.display import HTML, display
    display(HTML(html_doc))
except Exception:
    pass

MISTRA 2026 ANNUAL PGM ROUNDTABLE — EXECUTION DASHBOARD KPIs
Pipeline capex (ZAR bn)         :      269.5
Pipeline H2 capacity (MW)       :       6740
PGM feedstock (koz/yr)          :       2291
Jobs supported                  :      42870
Weighted average IRR (%)        :       16.7
Assets in register              :         18

Saved standalone dashboard -> mistra_2026_pgm_hydrogen_dashboard.html  (0.53 MB)
